# Olist Sales & Customer Behavior Analysis

**English edition.** The narrative (markdown) cells are translated from the Persian original; the code cells are identical to the Persian edition, so running the notebook reproduces exactly the same numbers, tables and charts. Table and chart titles rendered by the code remain in Persian.

Raw data is read from the `Olist-data/` folder. Required libraries: `pandas`, `numpy`, `matplotlib`, `seaborn`, `packaging`, `arabic-reshaper`, `python-bidi`. Run all cells top-to-bottom from the project root. Delivered merchandise value covers item price only and excludes freight.


# 1. Project Introduction

In this project we act as a Data Analyst for an e-commerce platform.

The data belongs to **Olist**, a Brazilian e-commerce platform that lets many sellers offer their products through online storefronts to customers across Brazil.

Over its lifetime the platform has recorded a large volume of information about orders, customers, products, sellers, payments, shipping and customer reviews.

Data by itself has little business value. Real value is created when we can use it to answer business questions.

In this project we move from raw data to information, patterns and finally business insight using real data.

## Project Goals

Imagine we have just joined the Olist data team and the managers have asked us to examine the available data and present an overall picture of the business.

By the end of the project we should be able to tell management:

- What state is the business in?
- How do customers behave?
- Which parts perform well?
- Which parts may need further investigation?

### Skills Used

- Data cleaning and preparation
- Working with multiple `DataFrame`s
- Joining tables
- `groupby` and aggregation
- Working with dates and times
- Deriving business KPIs
- Customer behaviour analysis
- Using `Pandas` for data analysis
- Choosing appropriate charts
- Plotting with `Matplotlib` and `Seaborn`
- Interpreting charts
- Extracting business insight


### Four Management Questions

- **Business state:** orders and delivered merchandise value are compared over matched months.
- **Customer behaviour:** geographic concentration and the second-order rate are measured on an equal-observation basis.
- **Strengths:** the share of early deliveries and of high review scores are the evaluation basis.
- **Areas to review:** the late-delivery rate and the review-score gap have top priority.


# 2. Data Introduction

Unlike many simple analysis exercises, this dataset is not a single `CSV` file. Business information is stored in several tables, each describing one part of the store's activity.

## Tables in the Dataset

| Table | Content |
| :---- | :------ |
| Orders | order status, purchase, shipping and delivery timestamps |
| Customers | customer information and location |
| Products | product information and category |
| Order items | purchased product, seller and price |
| Payments | payment method, amount and instalment count |
| Sellers | seller information and location |
| Reviews | the score a customer leaves after purchase |

Answering many project questions requires combining tables through shared keys.

## Dataset Structure

The dataset structure and the relationships between tables are shown in the figure below (next cell).


### Data Structure

Figure 2.1 shows how the tables and keys relate. The category-translation table connects to products through `product_category_name`. The geolocation zip-prefix key is not unique, so joining it directly to orders can multiply the row count.

![Table structure of the Olist data](attachment:database_structure.png)


In [ ]:
%matplotlib inline
from pathlib import Path
import re
from html import escape
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import ft2font
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from packaging.version import Version
import arabic_reshaper
from bidi.algorithm import get_display
from IPython.display import display


def fa(value):
    value = str(value)
    value = re.sub(r'(?<![0-9])\d{4}-\d{2}(?:-\d{2})?(?![0-9])',
                   lambda m: '‎' + m.group() + '‎', value)
    if Version(matplotlib.__version__) >= Version('3.11') and getattr(ft2font, '__libraqm_version__', ''):
        return value
    return '\n'.join(get_display(arabic_reshaper.reshape(line)) for line in value.split('\n'))


class RTLTable:
    def __init__(self, caption, frame):
        self.caption, self.frame = caption, frame

    def __repr__(self):
        return self.caption + "\n" + self.frame.to_string()

    def _repr_html_(self):
        table = self.frame._repr_html_().replace('<table ', '<table dir="rtl" ', 1)
        table = re.sub(r'<(th|td)(?=[ >])', lambda m: '<' + m.group(1) + ' dir="auto"', table)
        return '<div dir="rtl" style="text-align: right;"><p>' + escape(self.caption) + '</p>' + table + '</div>'


def show_table(number, title, frame, formats=None, labels=None, sample=None, period=None):
    out = frame.to_frame() if isinstance(frame, pd.Series) else frame.copy()
    assert out.shape[1] + out.index.nlevels <= 6, (number, out.shape)
    formats, labels = formats or {}, labels or {}
    for col in out.columns:
        kind = formats.get(col, '')
        def fmt(v):
            if pd.isna(v): return '—'
            if kind == 'brl': return f'{(0.0 if abs(v) < 0.005 else v):,.2f} BRL'
            if kind == 'pct': return f'{v:,.1f}%'
            if kind == 'score': return f'{v:.2f}'
            if kind == 'pp': return f'{v:.1f}'
            if kind == 'decimal': return f'{v:,.2f}'
            if isinstance(v, (bool, np.bool_)): return str(bool(v))
            if isinstance(v, (int, np.integer)): return f'{v:,}'
            if isinstance(v, (float, np.floating)): return f'{v:,.2f}'
            return str(v)
        out[col] = out[col].map(fmt)
    out = out.rename(columns={key: SHORT_LABELS.get(value, value) for key, value in labels.items()})
    assert out.columns.is_unique, number
    caption = f'جدول {number} | {title} | {sample} | {period or ORDER_PERIOD}'
    display(RTLTable(caption, out))


def heading(ax, number, title, period=None):
    title = title.replace('، کل دوره داده', '').replace('، کل دوره', '')
    ax.set_title(fa(f'نمودار {number} | {title}، {period or ORDER_PERIOD}'), fontsize=10, pad=12)


def axes_labels(ax, x, y):
    ax.set_xlabel(fa(x))
    ax.set_ylabel(fa(y))


def complete():
    plt.tight_layout()
    plt.show()


sns.set_theme(style='whitegrid', font='DejaVu Sans', rc={
    'axes.spines.top': False, 'axes.spines.right': False,
    'grid.color': '#e2ebeb', 'axes.edgecolor': '#bfd0d1',
    'axes.unicode_minus': False, 'figure.dpi': 110,
})
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 180)
DATA = Path('Olist-data')
files = {
    'orders': 'olist_orders_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'translation': 'product_category_name_translation.csv',
}
missing = [str(DATA / f) for f in files.values() if not (DATA / f).is_file()]
if missing:
    raise FileNotFoundError('Missing raw input files: ' + ', '.join(missing))
zip_types = {
    'customers': {'customer_zip_code_prefix': 'string'},
    'sellers': {'seller_zip_code_prefix': 'string'},
    'geolocation': {'geolocation_zip_code_prefix': 'string'},
}
raw = {name: pd.read_csv(DATA / path, dtype=zip_types.get(name), encoding='utf-8-sig')
       for name, path in files.items()}
START = pd.Timestamp('2017-01-01')
END = pd.Timestamp('2018-09-01')
MIN_REVIEWS = 30
MIN_STATE = 100
SCORE_GAP = 0.10
WINDOW_DAYS = 90
FIRST_YEAR = START.year
LAST_YEAR = (END - pd.Timedelta(days=1)).year
TREND_PERIOD = f"{START:%Y-%m} تا {(END - pd.Timedelta(days=1)):%Y-%m}"
ORDER_DATES = pd.to_datetime(raw['orders'].order_purchase_timestamp, errors='coerce')
ORDER_PERIOD = f"{ORDER_DATES.min():%Y-%m-%d} تا {ORDER_DATES.max():%Y-%m-%d}"
REVIEW_DATES = pd.to_datetime(raw['reviews'].review_creation_date, errors='coerce')
REVIEW_PERIOD = f"{REVIEW_DATES.min():%Y-%m-%d} تا {REVIEW_DATES.max():%Y-%m-%d}"
COHORT_END = (END - pd.Timedelta(days=WINDOW_DAYS)).replace(day=1)
COHORT_PERIOD = f"{START:%Y-%m} تا {(COHORT_END - pd.Timedelta(days=1)):%Y-%m}"
SHORT_LABELS = {
    'ارزش کالا، سفارش‌های تحویل‌شده، BRL':'ارزش تحویل‌شده',
    'ارزش کالا، تحویل‌شده':'ارزش تحویل‌شده', 'ارزش کالا، همه وضعیت‌ها':'ارزش همه وضعیت‌ها',
    'میانگین امتیاز نظرها':'امتیاز نظرها', 'میانگین امتیاز در سطح سفارش':'امتیاز سفارش',
    'مشتری با فرصت مشاهده':'مشتری واجد فرصت', 'مشتری با سفارش دوم':'سفارش دوم',
    'تعداد فروشنده از ده فروشنده نخست':'فروشنده برتر', 'در فهرست نامزدها':'نامزد',
    'سفارش پرداخت‌شده بدون قلم':'پرداخت مثبت بی‌قلم', 'سفارش بدون قلم ثبت‌شده':'سفارش بی‌قلم',
    'اختلاف رتبه ارزش و تعداد':'اختلاف رتبه', 'فاصله امتیاز زودتر و دیرتر':'فاصله امتیاز',
    'میانگین تعداد اقساط':'میانگین اقساط', 'سفارش چندتراکنشی':'چندتراکنشی',
    'فاصله از گروه بدون تأخیر':'فاصله امتیاز', 'فاصله از مبنای نظرها':'فاصله امتیاز',
    'تغییر نسبت به دوره نخست':'درصد تغییر', 'تعداد سفارش قابل مقایسه':'سفارش قابل مقایسه',
    'دارای قلم در همه وضعیت‌ها':'فروشنده دارای قلم', 'دارای قلم تحویل‌شده':'فروشنده تحویل‌شده',
    'تعداد نامزد سخت‌گیرانه':'نامزد سخت‌گیرانه', 'تعداد نامزد گسترده':'نامزد گسترده',
    'آستانه ارزش گروه واجد شرط':'آستانه واجد شرط', 'آستانه ارزش همه فروشندگان تحویل‌شده':'آستانه همه',
}


# 3. Data Understanding

Before analysis, we need to understand the data at hand.

## Questions & Tasks

1. Which files and tables exist in the dataset?
2. How many rows and columns does each table have?
3. What does one row represent in each table?
4. Identify the important columns of each table.
5. Check column `dtype`s — does any data type look inappropriate?
6. Which columns can link the tables together?
7. Is there duplicate data in the tables?
8. Check missing values in each table.
9. Compute the percentage of missing values per column.

## Expected Output

At the end of this section, explain in a few lines the overall structure of the data and the relationships between tables.


### 3.1. Table Size and Row Grain

Table 3.1 shows each table's size, row grain and join key. Table 3.2 reports full-row duplicates and missing cells without removing rows.


In [ ]:
grains = {
    'orders': ('سفارش', 'order_id'),
    'customers': ('مشتری در یک سفارش', 'customer_id'),
    'items': ('قلم در سفارش', 'order_id + order_item_id'),
    'payments': ('تراکنش پرداخت', 'order_id + payment_sequential'),
    'reviews': ('رکورد نظر', 'order_id / review_id'),
    'products': ('محصول', 'product_id'),
    'sellers': ('فروشنده', 'seller_id'),
    'geolocation': ('رکورد مکان و کد پستی', 'geolocation_zip_code_prefix'),
    'translation': ('ترجمه کلید دسته', 'product_category_name'),
}
inventory = pd.DataFrame([
    {'table_key': name, 'file': files[name], 'rows': len(df), 'columns': df.shape[1],
     'grain': grains[name][0], 'key': grains[name][1]}
    for name, df in raw.items()
]).set_index('table_key')
show_table('3.1', 'اندازه و کلید جدول‌های خام', inventory, labels={
    'file':'نام فایل', 'rows':'تعداد ردیف', 'columns':'تعداد ستون', 'grain':'واحد ردیف', 'key':'کلید'}, sample=f'{len(raw):,} جدول خام', period=ORDER_PERIOD)
quality = pd.DataFrame([
    {'table_key': name, 'duplicates': int(df.duplicated().sum()),
     'missing_cells': int(df.isna().sum().sum())}
    for name, df in raw.items()
]).set_index('table_key')
show_table('3.2', 'تکرار کامل سطر و مقدار گمشده', quality,
      labels={'duplicates':'سطر تکراری کامل', 'missing_cells':'خانه گمشده'}, sample=f'{len(raw):,} جدول خام', period=ORDER_PERIOD)

### 3.2. Data Types and Missing Values

Table 3.3 shows the dtype, count and share of missing values per column. Table 3.4 separates the date columns that need conversion. Zip codes are read as strings so leading zeros survive.


In [ ]:
profiles = []
for name, df in raw.items():
    for col in df.columns:
        profiles.append({'table_key':name, 'column':col, 'dtype':str(df[col].dtype),
                         'missing':int(df[col].isna().sum()),
                         'missing_pct':df[col].isna().mean()*100})
column_profile = pd.DataFrame(profiles).set_index(['table_key', 'column'])
show_table('3.3', 'نوع داده و پوشش ستون‌ها', column_profile, {'missing_pct':'pct'},
      {'dtype':'نوع خوانده‌شده', 'missing':'تعداد گمشده', 'missing_pct':'سهم گمشده'}, sample=f'{len(column_profile):,} ستون', period=ORDER_PERIOD)
time_cols = {
    'orders': ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
               'order_delivered_customer_date', 'order_estimated_delivery_date'],
    'items': ['shipping_limit_date'],
    'reviews': ['review_creation_date', 'review_answer_timestamp'],
}
time_profile = pd.DataFrame([
    {'table_key': name, 'column': col, 'raw_dtype': str(raw[name][col].dtype),
     'invalid': int((pd.to_datetime(raw[name][col], errors='coerce').isna() & raw[name][col].notna()).sum())}
    for name, cols in time_cols.items() for col in cols
]).set_index(['table_key', 'column'])
show_table('3.4', 'ستون‌های زمانی و تبدیل‌ناپذیرها', time_profile,
      labels={'raw_dtype':'نوع خام', 'invalid':'مقدار تبدیل‌ناپذیر'}, sample=f'{len(time_profile):,} ستون زمانی', period=ORDER_PERIOD)

### 3.3. Keys and Structural Summary

Table 3.5 checks key uniqueness. The order key connects an order to its items, payments and reviews; the customer key carries each order to the stable customer identity. One order can have several items, payments or reviews, so joining every table directly without defining the analysis unit inflates row counts.

The zip prefix is not unique in the geolocation table; for that reason it is not joined directly to orders. The products table links to the translation table through the category key. In table 3.4 dates appear as strings; converting them to a datetime type is required for duration and trend calculations, and a missing delivery date is never treated as "delivered on the estimated date".


In [ ]:
key_checks = pd.DataFrame([
    {'key':'orders.order_id', 'unique':raw['orders'].order_id.is_unique},
    {'key':'customers.customer_id', 'unique':raw['customers'].customer_id.is_unique},
    {'key':'items.order_id + order_item_id', 'unique':not raw['items'].duplicated(['order_id','order_item_id']).any()},
    {'key':'payments.order_id + payment_sequential', 'unique':not raw['payments'].duplicated(['order_id','payment_sequential']).any()},
    {'key':'reviews.review_id', 'unique':raw['reviews'].review_id.is_unique},
    {'key':'reviews.order_id', 'unique':raw['reviews'].order_id.is_unique},
    {'key':'geolocation.geolocation_zip_code_prefix', 'unique':raw['geolocation'].geolocation_zip_code_prefix.is_unique},
]).set_index('key')
show_table('3.5', 'بررسی یکتایی کلیدها', key_checks, labels={'unique':'کلید یکتا'}, sample=f'{len(key_checks):,} کلید', period=ORDER_PERIOD)

### Summary of Data Structure and Table Relationships

Each `orders` row is one order and connects through `order_id` to items (`order_id + order_item_id`), payment transactions (`order_id + payment_sequential`) and reviews. `customers.customer_id` is unique per order and carries the order to the stable `customer_unique_id`; `items` connects to products and sellers through `product_id` and `seller_id`, and products connect to the category translation table through `product_category_name`. Three points govern table combinations: (1) items and payments have several rows per order, so aggregation must happen at the analysis-unit level; (2) in this data edition `review_id` (814 duplicated rows) and the reviews table's `order_id` (551 extra rows) are not unique, so a direct review join would fan out rows — review scores are therefore averaged per order; (3) the geolocation table holds 1,000,163 rows for 19,015 zip prefixes (261,831 exact duplicate rows), and joining it directly to orders would multiply rows, so the state is taken from the customers table.


# 4. Data Cleaning & Preparation

Prepare the data for business analysis.

## Questions & Tasks

1. What do you decide about the observed missing values? Explain your reasoning.
2. Check duplicate records.
3. Fix column data types.
4. Convert date/time columns to `datetime`.
5. Check unusual or suspicious values.
6. Create new columns where needed, such as: order year, order month, order day, delivery duration, days of delivery delay.

## Expected Output

By the end of this section you should have clean `DataFrame`s ready for analysis.


### 4.1. Cleaning Decisions and Ready Tables

Dates are converted to datetime. Orders, items and reviews are kept without dropping duplicate rows. Exactly duplicated geolocation records are separated only in a dedicated geolocation set. Duration and delay-rate calculations use only valid dates and non-negative durations. Tables 4.1 and 4.2 show the exceptions and the decisions.


In [ ]:
orders = raw['orders'].copy()
items = raw['items'].copy()
customers = raw['customers'].copy()
payments = raw['payments'].copy()
reviews = raw['reviews'].copy()
products = raw['products'].copy()
sellers = raw['sellers'].copy()
geo = raw['geolocation'].copy()
translation = raw['translation'].copy()
for name, columns in time_cols.items():
    df = {'orders':orders, 'items':items, 'reviews':reviews}[name]
    for col in columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
orders['purchase_month'] = orders.order_purchase_timestamp.dt.to_period('M')
orders['purchase_year'] = orders.order_purchase_timestamp.dt.year
orders['purchase_weekday'] = orders.order_purchase_timestamp.dt.day_of_week
geo_unique = geo.drop_duplicates().copy()
geo_outside = (~geo.geolocation_lat.between(-34, 6) | ~geo.geolocation_lng.between(-74, -28))
order_customer = orders.merge(customers, on='customer_id', how='left', validate='one_to_one')
delivered_orders = orders.loc[orders.order_status.eq('delivered')].copy()
duration_orders = delivered_orders.dropna(subset=['order_purchase_timestamp', 'order_delivered_customer_date']).copy()
duration_orders['duration_days'] = (duration_orders.order_delivered_customer_date - duration_orders.order_purchase_timestamp).dt.total_seconds()/86400
duration_orders = duration_orders.loc[duration_orders.duration_days.ge(0)].copy()
delivery = duration_orders.dropna(subset=['order_estimated_delivery_date']).copy()
delivery['delay_days'] = (delivery.order_delivered_customer_date.dt.normalize() -
                          delivery.order_estimated_delivery_date.dt.normalize()).dt.days
delivery['delivery_group'] = np.select([delivery.delay_days.lt(0), delivery.delay_days.eq(0)],
                                       ['early', 'on_time'], default='late')
delivery['is_late'] = delivery.delay_days.gt(0)
all_items = items.merge(orders[['order_id','order_status','order_purchase_timestamp','purchase_month']],
                        on='order_id', how='left', validate='many_to_one')
all_items = all_items.merge(products[['product_id','product_category_name']],
                            on='product_id', how='left', validate='many_to_one')
all_items = all_items.merge(translation[['product_category_name','product_category_name_english']],
                            on='product_category_name', how='left', validate='many_to_one')
all_items['category'] = (all_items.product_category_name_english.fillna(all_items.product_category_name)
                         .fillna('unknown'))
del_items = all_items.loc[all_items.order_status.eq('delivered')].copy()
review_order = reviews.groupby('order_id').review_score.mean().rename('order_mean_score')
review_baseline = reviews.review_score.mean()
order_baseline = review_order.mean()
exception_table = pd.DataFrame([
    {'case':'delivered_without_date', 'count':int(delivered_orders.order_delivered_customer_date.isna().sum()),
     'decision':'کنارگذاری از تحلیل تحویل'},
    {'case':'other_status_with_delivery_date', 'count':int((~orders.order_status.eq('delivered') & orders.order_delivered_customer_date.notna()).sum()),
     'decision':'نگهداری، خارج از تحلیل تحویل'},
    {'case':'not_defined', 'count':int(payments.payment_type.eq('not_defined').sum()),
     'decision':'نگهداری تراکنش ثبت‌شده'},
]).set_index('case')
show_table('4.1', 'وضعیت‌های دارای تاریخ یا روش نامعمول', exception_table,
      labels={'count':'تعداد', 'decision':'تصمیم'}, sample=f'{len(orders):,} سفارش و {len(payments):,} تراکنش', period=ORDER_PERIOD)
quality_decisions = pd.DataFrame([
    {'check':'geolocation_full_duplicates', 'count':int(geo.duplicated().sum()), 'decision':'حذف در نسخه بررسی جغرافیا'},
    {'check':'geolocation_outside_bounding_box', 'count':int(geo_outside.sum()), 'decision':'نگهداری خام، بررسی مجزا'},
    {'check':'delivery_over_100_days', 'count':int(duration_orders.duration_days.gt(100).sum()), 'decision':'نگهداری، مقدار دورافتاده ممکن'},
    {'check':'delivery_negative_days', 'count':int(((delivered_orders.order_delivered_customer_date-delivered_orders.order_purchase_timestamp).dt.total_seconds()<0).sum()), 'decision':'خارج از تحلیل مدت'},
    {'check':'product_weight_zero', 'count':int(products.product_weight_g.eq(0).sum()), 'decision':'نگهداری، خارج از تحلیل وزن معتبر'},
    {'check':'item_price_zero', 'count':int(items.price.eq(0).sum()), 'decision':'نگهداری و گزارش'},
    {'check':'freight_value_zero', 'count':int(items.freight_value.eq(0).sum()), 'decision':'نگهداری، ممکن است حمل رایگان باشد'},
]).set_index('check')
show_table('4.2', 'تکرار و مقدارهای غیرمعمول', quality_decisions,
      labels={'count':'تعداد', 'decision':'تصمیم'}, sample=f'{len(quality_decisions):,} شاخص کیفیت', period=ORDER_PERIOD)

The bounding box in table 4.2 is a first-pass coordinate test; being inside it does not prove a point lies on Brazil's real border. Far outliers are kept in the raw data and every analysis decision is recorded in the table.


# 5. Orders & Business-Activity Trend

Olist managers want to know how the store's activity volume changed over time and whether there is a recognisable pattern in when orders are placed.

## Questions & Tasks

### 5.1. Order Status
1. Compute the total number of recorded orders. How many were delivered, canceled or in other states?
2. Compute the count and share of orders by `order_status`, displaying them so the share of delivered, canceled and other states is immediately visible.

### 5.2. Monthly Trend
3. Compute orders per month; plot the monthly trend and identify growth, decline or volatility periods. Which month had the most and which the fewest orders?
4. Break orders down by year and month. Has the store's activity grown over the data period? Support the answer with a time plot and numeric evidence.

### 5.3. Weekly Pattern
5. Compute orders by weekday. On which days do customers order more? Is the difference between days significant?

### 5.4. High-Activity Months
6. Name the five months with the most orders. Do they belong to one year or are they spread out?

## Business Conclusion

Given the order trend, what changed in Olist's activity level? Is the evidence sufficient to speak of sustained growth or a seasonal pattern?


### 5.1. Order Status

Table 5.1 covers every recorded status. Chart 5.1 makes the per-status comparison easy.


In [ ]:
status_summary = orders.order_status.value_counts().rename_axis('order_status').to_frame('orders')
status_summary['share_pct'] = status_summary.orders.div(len(orders)).mul(100)
show_table('5.1', 'تعداد و سهم وضعیت سفارش‌ها', status_summary, {'share_pct':'pct'},
      {'orders':'تعداد سفارش','share_pct':'سهم از کل'}, sample=f'{len(orders):,} سفارش', period=ORDER_PERIOD)
fig, ax = plt.subplots(figsize=(8.8,4.0))
ax.barh(status_summary.index, status_summary.orders, color='#4b8d99')
ax.invert_yaxis()
heading(ax,'5.1',f'وضعیت سفارش‌ها، {len(orders):,} سفارش')
axes_labels(ax,'تعداد سفارش','کلید وضعیت')
complete()

Each status share is recorded in table 5.1. The delivered share next to the other statuses gives a picture of order completion.


In table 5.2 months outside the core window are flagged as incomplete; a small count alone does not prove a month is incomplete. Chart 5.2 covers the continuous window. Table 5.3 separates the two matched periods from the middle months.


In [ ]:
monthly_observed = orders.groupby('purchase_month').size().sort_index()
monthly_all = monthly_observed.reindex(pd.period_range(monthly_observed.index.min(), monthly_observed.index.max(), freq='M'), fill_value=0)
monthly = monthly_all.rename_axis('month').to_frame('orders')
monthly['scope'] = np.where((monthly.index >= START.to_period('M')) & (monthly.index < END.to_period('M')),
                            'بازه تحلیل', 'ناقص')
show_table('5.2', 'تعداد ثبت ماهانه و پوشش دوره', monthly, labels={'orders':'تعداد سفارش','scope':'پوشش'}, sample=f'{len(orders):,} سفارش', period=ORDER_PERIOD)
core_monthly = monthly_all.loc[(monthly_all.index >= START.to_period('M')) & (monthly_all.index < END.to_period('M'))]
orders_2017 = int(core_monthly.loc['2017-01':'2017-08'].sum())
orders_2018 = int(core_monthly.loc['2018-01':'2018-08'].sum())
order_growth = (orders_2018 / orders_2017 - 1) * 100
mid_orders = int(core_monthly.sum()) - orders_2017 - orders_2018
period_orders = pd.DataFrame({'period':['دوره اول هم‌ماه','دوره دوم هم‌ماه','دو دوره هم‌ماه','ماه‌های میانی','کل بازه پیوسته'],
                              'orders':[orders_2017,orders_2018,orders_2017+orders_2018,mid_orders,int(core_monthly.sum())],
                              'growth_pct':[np.nan,order_growth,np.nan,np.nan,np.nan]}).set_index('period')
assert orders_2017 + orders_2018 + mid_orders == int(core_monthly.sum())
show_table('5.3','مقایسه ماه‌های مشترک و ماه‌های میانی',period_orders,{'growth_pct':'pct'},
      {'orders':'تعداد سفارش','growth_pct':'تغییر نسبت به دوره نخست'}, sample=f'{int(core_monthly.sum()):,} سفارش', period=TREND_PERIOD)
fig,ax=plt.subplots(figsize=(10,4.2))
ax.plot(core_monthly.index.to_timestamp(),core_monthly.values,marker='o',markersize=3,color='#287887',label=fa('سفارش ماهانه'))
ax.plot(core_monthly.index.to_timestamp(),core_monthly.rolling(3).mean().values,ls='--',color='#c67355',label=fa('میانگین متحرک سه ماه'))
heading(ax,'5.2',f'روند پیوسته شامل ماه‌های میانی، {int(core_monthly.sum()):,} سفارش',TREND_PERIOD)
axes_labels(ax,'ماه ثبت سفارش','تعداد سفارش');ax.legend(fontsize=8);plt.xticks(rotation=35);complete()

Chart 5.2 shows the continuous trend and table 5.3 compares the two matched periods. The first year's middle months appear only inside the continuous-window total. This distinction matters for capacity planning. The busiest month is November 2017 with 7,544 orders (table 5.5) — a spike coinciding with end-of-year campaigns such as Black Friday (24 November 2017). Together with the 135.1% growth between matched periods this is positive evidence of growth, but "sustained" growth still needs to be confirmed by watching the following months.


### 5.3. Weekdays and High-Activity Months

The weekday order in chart 5.3 is built from the date's day-of-week number. Table 5.4 shows per-day counts, table 5.5 the five busiest months and table 5.6 the peak and trough months of the analysis window.


In [ ]:
core_orders = orders.loc[orders.order_purchase_timestamp.ge(START) & orders.order_purchase_timestamp.lt(END)].copy()
weekday_names = ['دوشنبه','سه‌شنبه','چهارشنبه','پنجشنبه','جمعه','شنبه','یکشنبه']
weekday_counts = core_orders.purchase_weekday.value_counts().reindex(range(7),fill_value=0)
weekday_table = pd.DataFrame({'weekday':weekday_names,'orders':weekday_counts.to_numpy()}).set_index('weekday')
show_table('5.4','سفارش بر اساس روز هفته در دوره تحلیل',weekday_table,labels={'orders':'تعداد سفارش'}, sample=f'{len(core_orders):,} سفارش', period=TREND_PERIOD)
top_months=core_monthly.sort_values(ascending=False).head(5).rename_axis('month').to_frame('orders')
show_table('5.5','پنج ماه با بیشترین سفارش در دوره تحلیل',top_months,labels={'orders':'تعداد سفارش'}, sample=f'{len(core_orders):,} سفارش', period=TREND_PERIOD)
monthly_extremes=pd.DataFrame({'kind':['بیشترین','کمترین'],
    'month':[str(core_monthly.idxmax()),str(core_monthly.idxmin())],
    'orders':[int(core_monthly.max()),int(core_monthly.min())]}).set_index('kind')
show_table('5.6','بیشترین و کمترین ماه دوره تحلیل',monthly_extremes,labels={'month':'ماه','orders':'تعداد سفارش'}, sample=f'{len(core_orders):,} سفارش', period=TREND_PERIOD)
fig,ax=plt.subplots(figsize=(8.8,4))
ax.bar([fa(x) for x in weekday_names],weekday_counts.to_numpy(),color='#79a9a4')
heading(ax,'5.3',f'روز هفته، {len(core_orders):,} سفارش',TREND_PERIOD)
axes_labels(ax,'روز هفته','تعداد سفارش');plt.xticks(rotation=20);complete()

Monday has the most and Saturday the fewest orders in the selected window; the numbers are in table 5.4.

The weekly distribution is useful for support-shift planning and order-processing capacity.


# 6. Customers & Geographic Market

The marketing team wants to know where Olist customers are concentrated and how dependent the current market is on a few regions.

## Questions & Tasks

### 6.1. Customer Count
1. Compute the number of unique customers. When counting, mind the difference between row count and unique customers.

### 6.2. Customers by State
2. Compute customers per state; list the top ten states and show them in a ranked chart.
3. What share do the top five states hold? Compute each state's share and the five-state total. Is a large part of the customer base concentrated in a few states?

### 6.3. Customers by City
4. Identify the top ten cities. Compare with the top states. Is there also concentration in a few large cities?

### 6.4. Customers vs Orders
5. Compare each state's customer count with the orders placed from that state. Are there states with many customers whose order share is not proportional?
6. Show customers and orders in a comparison or relationship chart and highlight notable differences.

## Business Conclusion

In which regions is the Olist market concentrated? Which regions deserve further study for market development?

Note that a low customer count alone does not prove a growth opportunity.


### 6.1. Unique Customers and State Spread

The stable customer identity is used for unique counting. If a customer appears at several locations, charts 6.1 and tables 6.1–6.3 count the last recorded location, which is not necessarily the customer's current location.


In [ ]:
latest_customer = (order_customer.sort_values(['order_purchase_timestamp','order_id'])
                   .drop_duplicates('customer_unique_id',keep='last'))
unique_customers = int(order_customer.customer_unique_id.nunique())
state_customers = latest_customer.groupby('customer_state').customer_unique_id.nunique().sort_values(ascending=False)
state_summary = state_customers.to_frame('customers')
state_summary['share_pct'] = state_summary.customers.div(unique_customers).mul(100)
top5_state_share = state_summary.head(5).share_pct.sum()
customer_order_counts = order_customer.groupby('customer_unique_id').order_id.nunique()
single_customers = int(customer_order_counts.eq(1).sum())
customer_overview = pd.DataFrame({'unique_customers':[unique_customers],
                                  'single_order_customers':[single_customers],
                                  'single_order_pct':[single_customers/unique_customers*100],
                                  'top5_state_pct':[top5_state_share]})
show_table('6.1','مشتریان و تمرکز ایالتی',customer_overview,
      {'single_order_pct':'pct','top5_state_pct':'pct'},
      {'unique_customers':'مشتری یکتا','single_order_customers':'مشتری تک‌سفارشی',
       'single_order_pct':'سهم تک‌سفارشی','top5_state_pct':'سهم پنج ایالت نخست'}, sample=f'{unique_customers:,} مشتری', period=ORDER_PERIOD)
show_table('6.2','ده ایالت اول مشتریان یکتا',state_summary.head(10),{'share_pct':'pct'},
      {'customers':'مشتری یکتا','share_pct':'سهم مشتریان'}, sample=f'{unique_customers:,} مشتری', period=ORDER_PERIOD)
city_summary = (latest_customer.groupby(['customer_city','customer_state']).customer_unique_id.nunique()
                .sort_values(ascending=False).head(10).to_frame('customers'))
show_table('6.3','ده شهر اول مشتریان یکتا',city_summary,labels={'customers':'مشتری یکتا'}, sample=f'{unique_customers:,} مشتری', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.4,4.8))
st=state_summary.head(10)
ax.barh(st.index,st.customers,color='#438c94');ax.invert_yaxis()
heading(ax,'6.1',f'ده ایالت نخست، {unique_customers:,} مشتری یکتا، کل دوره')
axes_labels(ax,'تعداد مشتری یکتا','کلید ایالت');complete()

State shares and the customer totals are in tables 6.1 and 6.2.

Market concentration matters for prioritising services; few customers in a region do not by themselves prove a growth opportunity. The urban concentration question also has a positive answer: São Paulo alone holds 15.6% and Rio de Janeiro 6.9% of unique customers, and the top ten cities together hold 35.1% (table 6.3) — urban concentration is stronger than state concentration. Low-density North and Northeast states (e.g. PA, MA, CE) are candidates for market-development study, but purchasing power, logistics cost and repeat behaviour must be measured there first — not just the low customer count.


### 6.2. Customers vs Orders and Repeat Rate

In table 6.4 the order share and the customer share are not on the same basis: orders are counted by each order's location while customers are counted by their last recorded location. To avoid overlap, chart 6.2 does not repeat point labels; state names are in the table.


In [ ]:
state_orders = order_customer.groupby('customer_state').order_id.nunique().rename('orders')
state_comparison = state_summary.join(state_orders,how='outer').fillna(0)
state_comparison['order_pct'] = state_comparison.orders.div(len(orders)).mul(100)
state_comparison['gap_pp'] = state_comparison.order_pct - state_comparison.share_pct
state_comparison = state_comparison.sort_values('customers',ascending=False)
show_table('6.4','مشتری و سفارش به تفکیک ایالت',state_comparison,
      {'share_pct':'pct','order_pct':'pct','gap_pp':'pp'},
      {'customers':'مشتری یکتا','share_pct':'سهم مشتری','orders':'تعداد سفارش',
       'order_pct':'سهم سفارش','gap_pp':'اختلاف سهم، واحد درصد'}, sample=f'{len(order_customer):,} سفارش', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(7.8,4.8))
ax.scatter(state_comparison.customers,state_comparison.orders,s=58,alpha=.7,color='#387e8c')
heading(ax,'6.2',f'مشتری و سفارش ایالتی، {len(state_comparison):,} ایالت، کل دوره داده')
axes_labels(ax,'تعداد مشتری یکتا','تعداد سفارش');complete()

In chart 6.2 customer count and order count move together in most states. Table 6.4 shows the share gaps. The gap alone does not identify the cause of differing customer behaviour.


### 6.3. Second Order on an Equal-Observation Basis

Table 6.5 computes the second-order rate only for customers who had the same opportunity to order again. A customer's first order in the data is not necessarily their first real purchase.


In [ ]:
ranked_orders=(order_customer[['customer_unique_id','order_id','order_purchase_timestamp']]
               .dropna(subset=['customer_unique_id','order_purchase_timestamp'])
               .sort_values(['customer_unique_id','order_purchase_timestamp','order_id']).copy())
ranked_orders['position']=ranked_orders.groupby('customer_unique_id').cumcount()
first_order=ranked_orders.loc[ranked_orders.position.eq(0)].set_index('customer_unique_id').order_purchase_timestamp.rename('first_order')
second_order=ranked_orders.loc[ranked_orders.position.eq(1)].set_index('customer_unique_id').order_purchase_timestamp.rename('second_order')
cohorts=pd.concat([first_order,second_order],axis=1)
cohorts=cohorts.loc[cohorts.first_order.ge(START)&cohorts.first_order.lt(COHORT_END)].copy()
cohorts['repeat_90d']=cohorts.second_order.notna() & cohorts.second_order.le(cohorts.first_order+pd.Timedelta(days=WINDOW_DAYS))
cohorts['month']=cohorts.first_order.dt.to_period('M')
cohort_month=cohorts.groupby('month').repeat_90d.agg(customers='size',repeat_customers='sum')
cohort_month['repeat_pct']=cohort_month.repeat_customers.div(cohort_month.customers).mul(100)
repeat_90d_pct=cohorts.repeat_90d.mean()*100
cohort_total=pd.DataFrame({'customers':[len(cohorts)],'repeat_customers':[int(cohorts.repeat_90d.sum())],'repeat_pct':[repeat_90d_pct]})
show_table('6.5',f'سفارش دوم در {WINDOW_DAYS} روز',cohort_total,{'repeat_pct':'pct'},
      {'customers':'مشتری با فرصت مشاهده','repeat_customers':'مشتری با سفارش دوم','repeat_pct':'نرخ سفارش دوم'}, sample=f'{len(cohorts):,} مشتری', period=COHORT_PERIOD)
show_table('6.6','سفارش دوم به تفکیک ماه نخستین سفارش ثبت‌شده',cohort_month,{'repeat_pct':'pct'},
      {'customers':'تعداد مشتری','repeat_customers':'مشتری با سفارش دوم','repeat_pct':'نرخ سفارش دوم'}, sample=f'{len(cohorts):,} مشتری', period=COHORT_PERIOD)
fig,ax=plt.subplots(figsize=(9.2,4))
ax.plot(cohort_month.index.to_timestamp(),cohort_month.repeat_pct,marker='o',color='#287887')
ax.axhline(repeat_90d_pct,ls='--',color='#c67355',label=fa('نرخ کل گروه‌ها'))
heading(ax,'6.3',f'سفارش دوم در {WINDOW_DAYS} روز، {len(cohorts):,} مشتری',COHORT_PERIOD)
axes_labels(ax,'ماه نخستین سفارش ثبت‌شده','نرخ سفارش دوم، درصد');ax.legend(fontsize=8);plt.xticks(rotation=35);complete()

The second-order rate in table 6.5 has a different denominator and window from the full-period single-order share in table 6.1. For retention experiments this equal-window indicator is the more reliable one.


# 7. Products & Sales Performance

The sales team wants to know which product categories generate the most demand and which generate the most sales value.

## Questions & Tasks

### 7.1. Items Sold
1. Count the product categories and identify those with the most items sold. Show the top ten in a ranked chart.

### 7.2. Sales Value
2. Compute item sales value from the price column; aggregate by category, list the top ten categories by value and plot them in a separate chart.
3. Compare the ranking by items sold with the ranking by sales value. Are there categories with many items sold but a lower value rank? Give examples.

### 7.3. Average Price
4. Compute the average sold-item price per category. Find the five highest-priced categories. Are they necessarily among the best-sellers?

### 7.4. Category Share of Sales
5. Compute each category's share of total sales value. What percentage do the top five categories form?
6. Examine items sold and sales value together; draw a relationship chart to spot categories that perform differently on the two measures.

## Business Conclusion

Which categories matter by volume and which by value? Why can judging products by sales count alone be misleading?

**Note:** item sales value uses the price recorded in the order-items table and is not necessarily the company's net revenue.


### 7.1. Item Count and Category Value

The basis of tables 7.1, 7.2, 7.3, 7.4 and charts 7.1–7.2 is delivered order items only. Category names are the English keys. The unknown category stays in the total-value denominator. Categories without a translation entry keep their raw key in table 7.6; no replacement name was invented. Freight, costs and profit are not part of merchandise value.


In [ ]:
category_sales=(del_items.groupby('category',dropna=False)
                .agg(items=('order_item_id','size'),orders=('order_id','nunique'),
                     value_brl=('price','sum'),mean_price_brl=('price','mean')))
category_sales['item_rank']=category_sales['items'].rank(method='min',ascending=False).astype(int)
category_sales['value_rank']=category_sales.value_brl.rank(method='min',ascending=False).astype(int)
value_delivered=del_items.price.sum()
value_all=items.price.sum()
category_sales['share_pct']=category_sales.value_brl.div(value_delivered).mul(100)
seller_sales=(del_items.groupby('seller_id').agg(items=('order_item_id','size'),
              orders=('order_id','nunique'),value_brl=('price','sum')))
seller_sales['item_rank']=seller_sales['items'].rank(method='min',ascending=False).astype(int)
seller_sales['value_rank']=seller_sales.value_brl.rank(method='min',ascending=False).astype(int)
top_value_categories=category_sales.sort_values('value_brl',ascending=False)
top_item_categories=category_sales.sort_values('items',ascending=False)
top5_category_share=top_value_categories.head(5).share_pct.sum()
labels_cat={'items':'تعداد کالا','orders':'سفارش یکتا','value_brl':'ارزش کالا، سفارش‌های تحویل‌شده، BRL',
            'mean_price_brl':'میانگین قیمت، BRL','item_rank':'رتبه تعداد','value_rank':'رتبه ارزش','share_pct':'سهم ارزش'}
formats_cat={'value_brl':'brl','mean_price_brl':'brl','share_pct':'pct'}
show_table('7.1','ده دسته نخست بر اساس تعداد کالای تحویل‌شده',top_item_categories.head(10)[['items','orders','value_brl','item_rank','share_pct']],formats_cat,labels_cat, sample=f'{len(del_items):,} قلم', period=ORDER_PERIOD)
show_table('7.2','ده دسته نخست بر اساس ارزش کالای تحویل‌شده',top_value_categories.head(10)[['items','orders','value_brl','value_rank','share_pct']],formats_cat,labels_cat, sample=f'{len(del_items):,} قلم', period=ORDER_PERIOD)
comparison_names=top_item_categories.head(10).index.union(top_value_categories.head(10).index)
show_table('7.3','مقایسه رتبه تعداد و ارزش',category_sales.loc[comparison_names].sort_values('item_rank')[['items','orders','value_brl','item_rank','value_rank']],formats_cat,labels_cat, sample=f'{len(del_items):,} قلم', period=ORDER_PERIOD)
value_summary=pd.DataFrame({'delivered_brl':[value_delivered],'all_status_brl':[value_all],
                            'top5_pct':[top5_category_share]})
show_table('7.4','ارزش اصلی و مرجع همه وضعیت‌ها',value_summary,
      {'delivered_brl':'brl','all_status_brl':'brl','top5_pct':'pct'},
      {'delivered_brl':'ارزش کالا، تحویل‌شده','all_status_brl':'ارزش کالا، همه وضعیت‌ها','top5_pct':'سهم پنج دسته اول'}, sample=f'{len(del_items):,} قلم تحویل‌شده و {len(items):,} قلم همه وضعیت‌ها', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(10,5))
top=top_item_categories.head(10)
ax.barh(top.index,top['items'],color='#659daa');ax.invert_yaxis()
heading(ax,'7.1',f'تعداد کالا در دسته‌ها، تحویل‌شده، {len(del_items):,} قلم، کل دوره')
axes_labels(ax,'تعداد قلم کالا','کلید دسته');complete()

The item rank and value rank in table 7.3 are not identical. Using both criteria matters for assortment decisions; value alone does not reveal profit.


In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
top=top_value_categories.head(10)
ax.barh(top.index,top.value_brl,color='#be8a57');ax.invert_yaxis()
ax.xaxis.set_major_formatter(FuncFormatter(lambda v,pos:f'{v:,.0f}'))
heading(ax,'7.2',f'ده دسته نخست از نظر ارزش کالا، {len(del_items):,} قلم در مبنا')
axes_labels(ax,'ارزش کالا، سفارش‌های تحویل‌شده، BRL','کلید دسته');complete()

Top-category values are computed in table 7.2 and the top-five share of the total is in table 7.4.

Value concentration on a few categories raises the importance of checking their quality, but it does not show profit.


### 7.2. Average Price and Large-Seller Presence

Table 7.5 shows the five highest average-price categories together with their value rank and the number of top-ten sellers present in each. None of these five categories is in the top ten by value; the value rank and top-seller presence are in the same table.


In [ ]:
top10_seller_ids=seller_sales.nlargest(10,'value_brl').index
seller_top_presence=(del_items.loc[del_items.seller_id.isin(top10_seller_ids)]
                     .groupby('category').seller_id.nunique().rename('top_sellers'))
expensive=category_sales.sort_values('mean_price_brl',ascending=False).head(5).join(seller_top_presence).fillna({'top_sellers':0})
expensive['top_sellers']=expensive.top_sellers.astype(int)
show_table('7.5','قیمت متوسط بالا، رتبه ارزش و حضور فروشندگان برتر',expensive[['items','value_brl','mean_price_brl','value_rank','top_sellers']],
      formats_cat,{**labels_cat,'top_sellers':'تعداد فروشنده از ده فروشنده نخست'}, sample=f'{len(del_items):,} قلم', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8,5))
ax.scatter(category_sales['items'],category_sales.value_brl,alpha=.58,color='#588f9b',s=38)
heading(ax,'7.3',f'تعداد کالا و ارزش، تحویل‌شده، {len(category_sales):,} دسته، کل دوره')
axes_labels(ax,'تعداد قلم کالا','ارزش کالا، سفارش‌های تحویل‌شده، BRL')
ax.yaxis.set_major_formatter(FuncFormatter(lambda v,pos:f'{v:,.0f}'));complete()

Chart 7.3 shows the count–value relationship. Table 7.5 shows that a high average price alone is not enough to reach the top of total value; sales volume matters too.


### 7.3. Untranslated Keys in the Data

Table 7.7 compares merchandise value in the matched periods. Table 7.6 lists the raw category keys that have no English equivalent in the translation table. For these, no substitute name was created and their value remains inside the grand total.


In [ ]:
untranslated_categories=(del_items.loc[del_items.product_category_name.notna() & del_items.product_category_name_english.isna()]
    .groupby('product_category_name').agg(items=('order_item_id','size'),value_brl=('price','sum')))
show_table('7.6','دسته‌های فاقد معادل در جدول ترجمه، کلید خام',untranslated_categories,
      {'value_brl':'brl'},{'items':'تعداد کالا','value_brl':'ارزش کالا، تحویل‌شده'}, sample=f'{int(untranslated_categories["items"].sum()):,} قلم', period=ORDER_PERIOD)

core_del_items = del_items.loc[del_items.order_purchase_timestamp.ge(START) & del_items.order_purchase_timestamp.lt(END)].copy()
delivered_compare_n = int(core_del_items.order_purchase_timestamp.dt.month.lt(END.month).sum())
delivered_period_values = []
for year in (FIRST_YEAR, LAST_YEAR):
    lower = pd.Timestamp(year=year, month=START.month, day=START.day)
    upper = pd.Timestamp(year=year, month=END.month, day=END.day)
    delivered_period_values.append(float(core_del_items.loc[core_del_items.order_purchase_timestamp.ge(lower) &
                                                            core_del_items.order_purchase_timestamp.lt(upper),'price'].sum()))
delivered_growth = (delivered_period_values[1] / delivered_period_values[0] - 1) * 100
delivered_period_table = pd.DataFrame({'period':[str(FIRST_YEAR),str(LAST_YEAR)],
    'value_brl':delivered_period_values,'growth_pct':[np.nan,delivered_growth]}).set_index('period')
show_table('7.7','ارزش کالای تحویل‌شده در ماه‌های مشترک',delivered_period_table,
      {'value_brl':'brl','growth_pct':'pct'},{'value_brl':'ارزش تحویل‌شده','growth_pct':'تغییر ارزش'}, sample=f'{delivered_compare_n:,} قلم هم‌ماه', period=TREND_PERIOD)


# 8. Customer Payment Behaviour

The finance team wants to know which payment methods customers use and whether the purchase amount relates to the payment style.

## Questions & Tasks

### 8.1. Payment Methods
1. Compute the number of transactions per payment method. Which method is used most? Compute each method's share of all payment records and display the results.
2. Compute the total amount paid per method. Does the most-used method also carry the largest total?
3. Compute the average amount per method, compare them and identify which method goes with the higher average payment.

### 8.2. Instalments
4. Examine the distribution of `payment_installments`. How many payments had no instalment or a single one? Which instalment count is most common?
5. Examine the relationship between payment amount and instalment count with a chart that shows the spread and the likely relationship. Do higher amounts usually come with more instalments?
6. Break payments down by method and instalment count. Do different methods show different instalment patterns?

## Business Conclusion

Which methods do customers use most? Can the purchase amount be one factor related to choosing the instalment count?

**Note:** a row in the payments table is not necessarily one unique order; choose the counting method correctly.


### 8.1. Transactions and Payment Methods

Table 8.1 is at transaction level and covers all statuses; an order can have several transactions. `not_defined` and zero-amount transactions are kept. Average amount and zero-amount details are in table 8.9.


In [ ]:
payment_summary=payments.groupby('payment_type').agg(transactions=('order_id','size'),
    total_brl=('payment_value','sum'),mean_brl=('payment_value','mean'),
    mean_installments=('payment_installments','mean'),
    multiple_installments=('payment_installments',lambda s:int(s.gt(1).sum())),
    zero_value=('payment_value',lambda s:int(s.eq(0).sum()))).sort_values('transactions',ascending=False)
payment_summary['share_pct']=payment_summary.transactions.div(len(payments)).mul(100)
show_table('8.1','روش پرداخت، همه تراکنش‌ها',payment_summary[['transactions','total_brl','mean_installments','multiple_installments','share_pct']],
      {'total_brl':'brl','mean_brl':'brl','share_pct':'pct','mean_installments':'decimal'},
      {'transactions':'تعداد تراکنش','total_brl':'مبلغ کل','mean_brl':'میانگین مبلغ',
       'mean_installments':'میانگین تعداد اقساط','multiple_installments':'تراکنش چندقسطی',
       'zero_value':'تراکنش صفر','share_pct':'سهم تراکنش'}, sample=f'{len(payments):,} تراکنش', period=ORDER_PERIOD)

fig,ax=plt.subplots(figsize=(8.8,4))
share_sorted=payment_summary.sort_values('transactions')
ax.barh(share_sorted.index,share_sorted.transactions,color='#639e98')
for j,(idx,row) in enumerate(share_sorted.iterrows()):
    ax.text(row.transactions+len(payments)*0.012,j,f"{row.share_pct:.1f}%",va='center',fontsize=9)
heading(ax,'8.1',f'روش‌های پرداخت، {len(payments):,} تراکنش، کل دوره')
axes_labels(ax,'تعداد تراکنش','کلید روش پرداخت')
complete()


In [ ]:
installments=payments.payment_installments.value_counts().sort_index().rename_axis('installments').to_frame('transactions')
show_table('8.2','توزیع تعداد اقساط، همه تراکنش‌ها',installments,labels={'transactions':'تعداد تراکنش'}, sample=f'{len(payments):,} تراکنش', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(installments.index.astype(int),installments.transactions,color='#7daea9')
heading(ax,'8.2',f'تعداد اقساط، همه وضعیت‌ها، {len(payments):,} تراکنش، کل دوره')
axes_labels(ax,'تعداد اقساط','تعداد تراکنش');complete()

Chart 8.1 compares each method's transaction share: credit card leads with 73.9% and boleto follows with 19.0%. Charts 8.2 and table 8.2 show the instalment-count distribution. Table 8.1 shows that multi-instalment payments occur only under `credit_card`. Method differences matter for payment design but do not measure lost demand.


### 8.2. Amount and Instalments

Chart 8.3 shows positive-amount transactions with their count in the title; the amount axis is logarithmic so very large and small values are both visible. Table 8.3 separates the rank correlation for all transactions and for credit card. The coefficient summarises the direction and strength of rank association between −1 and +1; a log axis displays multiplicative distances.


In [ ]:
positive_pay=payments.loc[payments.payment_value.gt(0)].copy()
card_pay=payments.loc[payments.payment_type.eq('credit_card')]
payment_rho=payments.payment_value.rank().corr(payments.payment_installments.rank())
card_rho=card_pay.payment_value.rank().corr(card_pay.payment_installments.rank())
payment_relation=pd.DataFrame({'population':['all_transactions','credit_card'],
   'transactions':[len(payments),len(card_pay)],'rank_correlation':[payment_rho,card_rho]}).set_index('population')
show_table('8.3','رابطه رتبه‌ای مبلغ و اقساط',payment_relation,{'rank_correlation':'decimal'},
      {'transactions':'تعداد تراکنش','rank_correlation':'همبستگی رتبه‌ای'}, sample=f'{len(payments):,} تراکنش', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8,4.5))
ax.scatter(positive_pay.payment_installments,positive_pay.payment_value,s=5,alpha=.035,color='#397c8d')
ax.set_yscale('log')
heading(ax,'8.3',f'مبلغ و اقساط، مبلغ مثبت، {len(positive_pay):,} تراکنش، کل دوره')
axes_labels(ax,'تعداد اقساط','مبلغ تراکنش، BRL، محور لگاریتمی');complete()

The correlation in table 8.3 gives the direction and strength of the rank relationship.

A statistical relationship alone does not prove that changing instalments affects purchases or order value.


### 8.3. Reconciling Payments with Items and Freight

Tables 8.4–8.8 match payments, merchandise value and freight for all statuses at order level; item and payment rows are not multiplied together. The difference between payment and delivered merchandise value has a different definition and is recorded separately in table 8.4. Refund information and the cause of every difference are not present in the data.


In [ ]:
payment_by_order=payments.groupby('order_id').agg(paid_brl=('payment_value','sum'),transactions=('payment_value','size'))
item_by_order=items.groupby('order_id').agg(item_brl=('price','sum'),freight_brl=('freight_value','sum'))
reconcile=(orders[['order_id','order_status']].merge(payment_by_order,on='order_id',how='outer')
           .merge(item_by_order,on='order_id',how='outer'))
reconcile['without_item']=reconcile.item_brl.isna()
reconcile[['paid_brl','transactions','item_brl','freight_brl']]=reconcile[['paid_brl','transactions','item_brl','freight_brl']].fillna(0)
reconcile['difference_brl']=reconcile.paid_brl-reconcile.item_brl-reconcile.freight_brl
reconcile['paid_without_item']=reconcile.without_item & reconcile.paid_brl.gt(0)
reconcile['paid_without_item_brl']=reconcile.paid_brl.where(reconcile.without_item,0)
payment_total=payments.payment_value.sum()
freight_all=items.freight_value.sum()
payment_difference=reconcile.difference_brl.sum()
recon_total=pd.DataFrame({'payment_brl':[payment_total],'all_item_brl':[value_all],
                          'freight_brl':[freight_all],'difference_brl':[payment_difference],
                          'delivered_gap_brl':[payment_total-value_delivered]})
show_table('8.4','تطبیق کل، همه وضعیت‌ها و همه تراکنش‌ها',recon_total,
      {x:'brl' for x in recon_total.columns},
      {'payment_brl':'مجموع پرداخت','all_item_brl':'ارزش کالا، همه وضعیت‌ها',
       'freight_brl':'حمل ثبت‌شده','difference_brl':'اختلاف همه وضعیت‌ها',
       'delivered_gap_brl':'اختلاف با تحویل‌شده'}, sample=f'{len(payments):,} تراکنش', period=ORDER_PERIOD)
recon_status=reconcile.groupby('order_status',dropna=False).agg(
    orders=('order_id','size'),payment_brl=('paid_brl','sum'),item_brl=('item_brl','sum'),
    freight_brl=('freight_brl','sum'),difference_brl=('difference_brl','sum'),
    no_item_orders=('without_item','sum'),paid_without_item=('paid_without_item','sum'),
    paid_without_item_brl=('paid_without_item_brl','sum'))
show_table('8.5','ریشه‌یابی اختلاف بر اساس وضعیت سفارش',recon_status[['orders','payment_brl','item_brl','freight_brl','difference_brl']],
      {k:'brl' for k in ['payment_brl','item_brl','freight_brl','difference_brl','paid_without_item_brl']},
      {'orders':'سفارش','payment_brl':'پرداخت','item_brl':'کالا','freight_brl':'حمل',
       'difference_brl':'اختلاف','no_item_orders':'سفارش بدون قلم ثبت‌شده',
       'paid_without_item':'سفارش پرداخت‌شده بدون قلم','paid_without_item_brl':'پرداخت بدون قلم'}, sample=f'{len(reconcile):,} سفارش', period=ORDER_PERIOD)
alloc=payments.merge(reconcile[['order_id','paid_brl','difference_brl','transactions']],on='order_id',how='left',validate='many_to_one')
alloc['weight']=np.where(alloc.paid_brl.ne(0),alloc.payment_value.div(alloc.paid_brl),1/alloc.transactions)
alloc['allocated_brl']=alloc.difference_brl*alloc.weight
recon_type=alloc.groupby('payment_type').agg(transactions=('order_id','size'),allocated_brl=('allocated_brl','sum'))
no_payment=reconcile.loc[reconcile.transactions.eq(0),'difference_brl'].sum()
if abs(no_payment)>1e-8:
    recon_type.loc['without_payment']={'transactions':0,'allocated_brl':no_payment}
    recon_type['transactions']=recon_type['transactions'].astype('int64')
show_table('8.6','تخصیص اختلاف به روش‌های پرداخت',recon_type,{'allocated_brl':'brl'},
      {'transactions':'تعداد تراکنش','allocated_brl':'اختلاف تخصیص‌یافته'}, sample=f'{len(payments):,} تراکنش', period=ORDER_PERIOD)
no_item_status = recon_status[['no_item_orders','paid_without_item','paid_without_item_brl']]
show_table('8.7','سفارش‌های بدون قلم و پرداخت آن‌ها',no_item_status,{'paid_without_item_brl':'brl'},
      {'no_item_orders':'سفارش بی‌قلم','paid_without_item':'پرداخت مثبت بی‌قلم','paid_without_item_brl':'مبلغ بی‌قلم'}, sample=f'{len(reconcile):,} سفارش', period=ORDER_PERIOD)
payment_residual = recon_status.drop(index=['unavailable','canceled']).difference_brl.sum()
payment_parts = pd.DataFrame({'difference_brl':[recon_status.loc['unavailable','difference_brl'],
    recon_status.loc['canceled','difference_brl'], payment_residual]},
    index=['unavailable','canceled','other_statuses'])
assert np.isclose(payment_parts.difference_brl.sum(), payment_difference)
show_table('8.8','اجزای اختلاف پرداخت و کالای همه وضعیت‌ها همراه حمل',payment_parts,
      {'difference_brl':'brl'},{'difference_brl':'اختلاف'}, sample=f'{len(reconcile):,} سفارش', period=ORDER_PERIOD)
show_table('8.9','میانگین و مبلغ صفر به تفکیک روش پرداخت',payment_summary[['mean_brl','zero_value']],
      {'mean_brl':'brl'},{'mean_brl':'میانگین پرداخت','zero_value':'تراکنش صفر'}, sample=f'{len(payments):,} تراکنش', period=ORDER_PERIOD)


# 9. Shipping Performance

The operations team wants to evaluate delivery performance and locate regions or periods with more delay.

## Questions & Tasks

### 9.1. Delivery Duration
1. For delivered orders, compute the days between purchase and actual delivery. Exclude orders missing the required dates.
2. Compute mean, median, min and max delivery time. What does the mean–median gap say about the distribution?
3. Examine the delivery-time distribution with a histogram or similar chart. Do most orders arrive within a narrow range? Are there a few very long deliveries?

### 9.2. Delivery Delay
4. Compare the actual delivery date with the estimated date for every order; identify early, on-time and late deliveries. What percentage of delivered orders was late?
5. Compute the average delay days for late orders. How many orders were more than seven days late?

### 9.3. Geographic and Temporal Performance
6. Compute the average delivery time per customer state; identify the ten slowest states and compare them, taking each state's order count into account to avoid conclusions from tiny samples.
7. Track the average delivery time and the late-order percentage over time; does the trend indicate improving or worsening shipping performance?

## Business Conclusion

Is delay a pervasive problem or concentrated in particular regions and orders? Which indicators are more useful for monitoring shipping performance?


### 9.1. Actual Delivery Duration

Table 9.1 covers only delivered orders with valid dates and non-negative duration. Actual duration runs from purchase to delivery. Table 4.1 lists the orders removed from the denominator.


In [ ]:
duration_summary=pd.DataFrame({'orders':[len(duration_orders)],'mean_days':[duration_orders.duration_days.mean()],
    'median_days':[duration_orders.duration_days.median()],'min_days':[duration_orders.duration_days.min()],
    'max_days':[duration_orders.duration_days.max()]})
show_table('9.1','مدت واقعی تحویل سفارش‌های دارای تاریخ معتبر',duration_summary,
      {x:'decimal' for x in ['mean_days','median_days','min_days','max_days']},
      {'orders':'تعداد سفارش','mean_days':'میانگین، روز','median_days':'میانه، روز',
       'min_days':'کمینه، روز','max_days':'بیشینه، روز'}, sample=f'{len(duration_orders):,} سفارش', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(9,4.1))
ax.hist(duration_orders.duration_days,bins=np.arange(0,duration_orders.duration_days.max()+6,5),color='#6ea59e',edgecolor='white')
heading(ax,'9.1',f'توزیع مدت تحویل، {len(duration_orders):,} سفارش معتبر، کل دوره')
axes_labels(ax,'مدت واقعی تحویل، روز','تعداد سفارش');complete()

The mean duration in table 9.1 exceeds the median. The median is the middle value after sorting deliveries; a percentile is the boundary of part of the distribution. Both indicators help identify long deliveries.


### 9.2. Delivery vs Estimated Date

Table 9.2 and chart 9.3 use calendar days, not hours. Delivery on the promised day counts as "on time" and before it as "early"; the late-rate denominator is only comparable orders.


In [ ]:
group_keys=['early','on_time','late']
group_labels={'early':'زودتر','on_time':'به‌موقع','late':'دیرتر'}
delivery_groups=delivery.delivery_group.value_counts().reindex(group_keys,fill_value=0).rename_axis('delivery_group').to_frame('orders')
delivery_groups['share_pct']=delivery_groups.orders.div(len(delivery)).mul(100)
delivery_display=delivery_groups.copy();delivery_display.index=[group_labels[x] for x in delivery_display.index]
show_table('9.2','تحویل نسبت به روز تخمینی',delivery_display,{'share_pct':'pct'},
      {'orders':'تعداد سفارش','share_pct':'سهم سفارش قابل مقایسه'}, sample=f'{len(delivery):,} سفارش', period=ORDER_PERIOD)
late_count=int(delivery.is_late.sum());late_pct=delivery.is_late.mean()*100
late_over7=int(delivery.delay_days.gt(7).sum())
late_mean=delivery.loc[delivery.is_late,'delay_days'].mean()
late_summary=pd.DataFrame({'eligible':[len(delivery)],'late':[late_count],'late_pct':[late_pct],
    'mean_late_days':[late_mean],'over7':[late_over7]})
show_table('9.3','نرخ و شدت تأخیر',late_summary,{'late_pct':'pct','mean_late_days':'decimal'},
      {'eligible':'سفارش قابل مقایسه','late':'سفارش دیرتر','late_pct':'نرخ تأخیر',
       'mean_late_days':'میانگین تأخیر، روز','over7':'بیش از 7 روز تأخیر'}, sample=f'{len(delivery):,} سفارش', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8,3.8))
ax.barh([fa(group_labels[x]) for x in group_keys],delivery_groups.share_pct,color=['#55988b','#a6c5bd','#d2896a'])
ax.invert_yaxis()
for j,value in enumerate(delivery_groups.share_pct):ax.text(value+1,j,f'{value:.1f}%',va='center')
heading(ax,'9.2',f'تحویل نسبت به روز تخمینی، {len(delivery):,} سفارش، کل دوره')
axes_labels(ax,'سهم سفارش قابل مقایسه، درصد','وضعیت تحویل');complete()

Table 9.2 places the three-group counts and shares next to the rate and severity of delay in table 9.3. A long delivery time does not necessarily mean the promise was broken.


### 9.3. Geography and Delay Trend

Table 9.4 ranks states with the stated minimum sample. Table 9.6 places the matched periods and March 2018 side by side. Chart 9.3 covers only comparable orders inside the continuous window, so its denominator is smaller than the full period's.


In [ ]:
delivery_customer=delivery.merge(customers[['customer_id','customer_state']],on='customer_id',how='left',validate='many_to_one')
state_delivery=(delivery_customer.groupby('customer_state').agg(orders=('order_id','size'),
    late=('is_late','sum'),mean_days=('duration_days','mean'),median_days=('duration_days','median')))
state_delivery['late_pct']=state_delivery.late.div(state_delivery.orders).mul(100)
state_eligible=state_delivery.loc[state_delivery.orders.ge(MIN_STATE)].copy()
show_table('9.4',f'ایالت‌ها، حداقل {MIN_STATE} سفارش قابل مقایسه',
      state_eligible.sort_values('late_pct',ascending=False),
      {'late_pct':'pct','mean_days':'decimal','median_days':'decimal'},
      {'orders':'سفارش قابل مقایسه','late':'سفارش دیرتر','mean_days':'میانگین مدت، روز',
       'median_days':'میانه مدت، روز','late_pct':'نرخ تأخیر'}, sample=f'{int(state_eligible.orders.sum()):,} سفارش در ایالت‌های واجد شرط', period=ORDER_PERIOD)
core_delivery=delivery.loc[delivery.order_purchase_timestamp.ge(START)&delivery.order_purchase_timestamp.lt(END)].copy()
monthly_delivery=core_delivery.groupby('purchase_month').agg(orders=('order_id','size'),late=('is_late','sum'),mean_days=('duration_days','mean'))
monthly_delivery['late_pct']=monthly_delivery.late.div(monthly_delivery.orders).mul(100)
show_table('9.5','نرخ تأخیر و مدت ماهانه در دوره تحلیل',monthly_delivery,{'late_pct':'pct','mean_days':'decimal'},
      {'orders':'سفارش قابل مقایسه','late':'سفارش دیرتر','late_pct':'نرخ تأخیر','mean_days':'میانگین مدت، روز'}, sample=f'{len(core_delivery):,} سفارش', period=TREND_PERIOD)
compare_delivery=[]
for year in (FIRST_YEAR,LAST_YEAR):
    sub=core_delivery.loc[core_delivery.order_purchase_timestamp.ge(f'{year}-01-01') &
                          core_delivery.order_purchase_timestamp.lt(f'{year}-09-01')]
    compare_delivery.append({'period':f'{year}-01 تا {year}-08','orders':len(sub),
                             'late':int(sub.is_late.sum()),'late_pct':sub.is_late.mean()*100})
march_delivery=core_delivery.loc[core_delivery.purchase_month.eq(pd.Period('2018-03'))]
compare_delivery.append({'period':'2018-03','orders':len(march_delivery),
                         'late':int(march_delivery.is_late.sum()),'late_pct':march_delivery.is_late.mean()*100})
delivery_compare=pd.DataFrame(compare_delivery).set_index('period')
delay_change_pp=delivery_compare.loc['2018-01 تا 2018-08','late_pct']-delivery_compare.loc['2017-01 تا 2017-08','late_pct']
show_table('9.6',f'دوره‌های هم‌ماه و مارس {LAST_YEAR}',delivery_compare,{'late_pct':'pct'},
      {'orders':'سفارش قابل مقایسه','late':'سفارش دیرتر','late_pct':'نرخ تأخیر'}, sample=f'{int(delivery_compare.iloc[0].orders+delivery_compare.iloc[1].orders):,} سفارش هم‌ماه', period=TREND_PERIOD)
show_table('9.7','تفاوت نرخ تأخیر دو دوره',pd.DataFrame({'change_pp':[delay_change_pp]}),
      {'change_pp':'pp'},{'change_pp':'تغییر نرخ، واحد درصد'}, sample=f'{int(delivery_compare.iloc[0].orders+delivery_compare.iloc[1].orders):,} سفارش هم‌ماه', period=TREND_PERIOD)
fig,ax=plt.subplots(figsize=(9.4,4))
ax.plot(monthly_delivery.index.to_timestamp(),monthly_delivery.late_pct,marker='o',color='#c16d51')
heading(ax,'9.3',f'روند نرخ تأخیر، {len(core_delivery):,} سفارش قابل مقایسه',TREND_PERIOD)
axes_labels(ax,'ماه ثبت سفارش','نرخ تأخیر، درصد');plt.xticks(rotation=35);complete()

The late rate of the two matched periods and of March is recorded with its denominator in table 9.6. Conclusion: delay is not a pervasive problem — it concentrates in particular regions and months. AL (21.4%), MA (17.4%) and SE (15.2%) top the state ranking (table 9.4) and the peak-sales months November 2017 (12.4%) and March 2018 (19.0%) top the monthly ranking (table 9.5), against a 6.8% overall average. For monitoring, the late rate and the actual delivery duration should be watched together and per route; a long duration alone does not mean a broken promise.


### 9.4. Trend of Actual Mean Delivery Duration

Chart 9.4 shows the mean actual delivery duration for the same months as table 9.5; chart 9.3 had shown the late rate separately.


In [ ]:
fig,ax=plt.subplots(figsize=(9.4,4))
ax.plot(monthly_delivery.index.to_timestamp(),monthly_delivery.mean_days,marker='o',color='#548b9a')
heading(ax,'9.4',f'مدت واقعی، {len(core_delivery):,} سفارش قابل مقایسه',TREND_PERIOD)
axes_labels(ax,'ماه ثبت سفارش','میانگین مدت تحویل، روز');plt.xticks(rotation=35);complete()

Table 9.5 shows the monthly mean duration and late rate. A falling late rate without a falling actual duration does not necessarily mean faster deliveries.


### 9.5. States with the Longest Delivery Duration

Table 9.8 and chart 9.5 show the states with the highest mean delivery duration after applying the stated minimum sample.


In [ ]:
slow_states=state_eligible.nlargest(10,'mean_days')
show_table('9.8',f'بیشترین میانگین مدت تحویل، حداقل {MIN_STATE} سفارش',slow_states,
      {'late_pct':'pct','mean_days':'decimal','median_days':'decimal'},
      {'orders':'سفارش قابل مقایسه','late':'سفارش دیرتر','mean_days':'میانگین مدت، روز',
       'median_days':'میانه مدت، روز','late_pct':'نرخ تأخیر'}, sample=f'{int(slow_states.orders.sum()):,} سفارش در ایالت‌های نمایش‌داده‌شده', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.5,4.5))
ax.barh(slow_states.index,slow_states.mean_days,color='#c58a63');ax.invert_yaxis()
heading(ax,'9.5',f'بیشترین مدت تحویل، {int(slow_states.orders.sum()):,} سفارش؛ حداقل {MIN_STATE} در هر ایالت')
axes_labels(ax,'میانگین مدت واقعی تحویل، روز','کلید ایالت');complete()

Table 9.8 puts each state's mean duration, late rate and sample size side by side. These three indicators together guide which routes deserve investigation.


# 10. Customer Satisfaction

The customer-experience team wants to know what scores customers leave and how satisfaction differs across product categories and over time.

## Questions & Tasks

### 10.1. Score Distribution
1. Count reviews per `review_score` and use a distribution chart to identify the most and least frequent scores.
2. Compute the mean score, the share of 4-or-5 reviews and the share of 1-or-2 reviews; interpret these indicators together.

### 10.2. Satisfaction by Category
3. Compute the mean score per product category; identify the ten highest and ten lowest categories and show them in a ranked chart.
4. Also check the review count per category. Do categories with very few reviews appear among the highest or lowest means? Explain why review count matters when interpreting a mean.

### 10.3. Satisfaction over Time
5. Compute the mean score over time, plot it and state whether a significant upward or downward trend exists.
6. Compare category mean scores with category sales value; identify categories with high sales value whose mean score is below the overall average.

## Business Conclusion

What is the overall satisfaction state? Which categories need more attention, and why should a category's performance not be judged on the mean score alone?


### 10.1. Score at Review Level

The main basis is the mean over all review records without dropping duplicated identifiers. The order-level mean is labelled separately in table 10.1; in this data the two means coincide at two decimals, but their analysis units differ.


In [ ]:
score_counts=reviews.review_score.value_counts().reindex(range(1,6),fill_value=0).rename_axis('review_score').to_frame('reviews')
score_counts['share_pct']=score_counts.reviews.div(len(reviews)).mul(100)
good_pct=reviews.review_score.ge(4).mean()*100
low_pct=reviews.review_score.le(2).mean()*100
score_summary=pd.DataFrame({'reviews':[len(reviews)],'review_mean_score':[review_baseline],
    'order_mean_score':[order_baseline],'positive_pct':[good_pct],'low_pct':[low_pct]})
show_table('10.1','مبنای امتیاز و سهم گروه‌ها',score_summary,
      {'review_mean_score':'score','order_mean_score':'score','positive_pct':'pct','low_pct':'pct'},
      {'reviews':'تعداد نظر','review_mean_score':'میانگین امتیاز نظرها',
       'order_mean_score':'میانگین امتیاز در سطح سفارش','positive_pct':'سهم امتیاز 4 و 5','low_pct':'سهم امتیاز 1 و 2'}, sample=f'{len(reviews):,} نظر', period=REVIEW_PERIOD)
show_table('10.2','توزیع امتیاز نظرها',score_counts,{'share_pct':'pct'},
      {'reviews':'تعداد نظر','share_pct':'سهم نظرها'}, sample=f'{len(reviews):,} نظر', period=REVIEW_PERIOD)
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(score_counts.index,score_counts.reviews,color='#639e98')
heading(ax,'10.1',f'امتیاز همه نظرها بر اساس تاریخ ثبت، {len(reviews):,} نظر',REVIEW_PERIOD)
axes_labels(ax,'امتیاز نظر','تعداد نظر');ax.set_xticks(range(1,6));complete()

Table 10.2 shows each score's share. The mean alongside the low-score share and the review volume gives a fuller picture of customer experience.


### 10.2. Category Scores with a Minimum Sample

Every review belongs to the whole order; if an order spans several categories it appears under each related category. The unit is therefore review–category and the review-count column does not sum to the total number of review records. Table 10.3 carries the minimum sample in its title. Chart 10.2 shows the highest and lowest categories from the same eligible population.


In [ ]:
order_category=del_items[['order_id','category']].drop_duplicates()
category_review_rows=order_category.merge(reviews[['order_id','review_score']],on='order_id',how='inner',validate='many_to_many')
category_rating=category_review_rows.groupby('category').agg(review_records=('review_score','size'),
    reviewed_orders=('order_id','nunique'),mean_score=('review_score','mean'))
category_metrics=category_sales.join(category_rating,how='left')
category_metrics['score_gap']=review_baseline-category_metrics.mean_score
category_eligible=category_metrics.loc[category_metrics.review_records.ge(MIN_REVIEWS)].copy()
category_ranked=category_eligible.sort_values('mean_score',ascending=False)
category_formats={'value_brl':'brl','mean_price_brl':'brl','share_pct':'pct','mean_score':'score','score_gap':'score'}
category_labels={**labels_cat,'review_records':'تعداد نظر','reviewed_orders':'سفارش نظردار',
                 'mean_score':'میانگین امتیاز نظرها','score_gap':'فاصله از مبنای نظرها'}
show_table('10.3',f'امتیاز دسته‌ها، حداقل {MIN_REVIEWS} نظر',
      category_ranked[['review_records','reviewed_orders','mean_score','score_gap','value_brl']],
      category_formats,category_labels, sample=f'{int(category_ranked.review_records.sum()):,} نظر–دسته واجد شرط', period=ORDER_PERIOD)
fig,axes=plt.subplots(1,2,figsize=(13,6.2))
for ax,part,desc,col in [(axes[0],category_ranked.head(10),'بالاتر','#57978d'),
                         (axes[1],category_ranked.tail(10).sort_values('mean_score'),'پایین‌تر','#c77b60')]:
    ax.barh(part.index,part.mean_score,color=col);ax.invert_yaxis();ax.set_xlim(0,5)
    ax.set_title(fa(desc));axes_labels(ax,'میانگین امتیاز نظرها','کلید دسته')
fig.suptitle(fa(f'نمودار 10.2 | دسته‌های بالا و پایین، {len(category_ranked.head(10))+len(category_ranked.tail(10)):,} دسته از {len(category_ranked):,} واجد شرط، حداقل {MIN_REVIEWS} نظر، {ORDER_PERIOD}'),fontsize=10)
complete()

The top and bottom ranks can be examined together with each category's review count in table 10.3; categories removed for small samples are listed separately in table 10.10.

A low score with a sufficient sample is the starting point for investigating review texts and the order process — not a verdict on the product.


### 10.3. Monthly Trend of Review Scores

This section's time axis is the review creation date, not the purchase date. Table 10.5 shows the monthly trend and table 10.6 compares the matched periods and March. Table 10.4 counts reviews outside the window.


In [ ]:
core_reviews=reviews.loc[reviews.review_creation_date.ge(START)&reviews.review_creation_date.lt(END)].copy()
review_coverage = pd.DataFrame({'reviews':[len(reviews),len(core_reviews),len(reviews)-len(core_reviews)]},
    index=['همه نظرها','در بازه روند','خارج از بازه روند'])
show_table('10.4','پوشش تاریخ ثبت نظر',review_coverage,labels={'reviews':'تعداد نظر'}, sample=f'{len(reviews):,} نظر', period=REVIEW_PERIOD)
core_reviews['review_month']=core_reviews.review_creation_date.dt.to_period('M')
monthly_reviews=core_reviews.groupby('review_month').agg(reviews=('review_score','size'),mean_score=('review_score','mean'))
show_table('10.5','روند امتیاز نظرها بر اساس تاریخ ثبت نظر',monthly_reviews,{'mean_score':'score'},
      {'reviews':'تعداد نظر','mean_score':'میانگین امتیاز نظرها'}, sample=f'{len(core_reviews):,} نظر', period=TREND_PERIOD)
review_period_rows=[]
for year in (FIRST_YEAR,LAST_YEAR):
    r=core_reviews.loc[core_reviews.review_creation_date.ge(f'{year}-01-01')&core_reviews.review_creation_date.lt(f'{year}-09-01')]
    review_period_rows.append({'period':f'{year}-01 تا {year}-08','reviews':len(r),'mean_score':r.review_score.mean()})
march_review=core_reviews.loc[core_reviews.review_month.eq(pd.Period('2018-03'))]
review_period_rows.append({'period':'2018-03','reviews':len(march_review),'mean_score':march_review.review_score.mean()})
review_compare=pd.DataFrame(review_period_rows).set_index('period')
show_table('10.6','امتیاز نظرها در دوره‌های هم‌ماه و مارس',review_compare,{'mean_score':'score'},
      {'reviews':'تعداد نظر','mean_score':'میانگین امتیاز نظرها'}, sample=f'{int(review_compare.iloc[:2].reviews.sum()):,} نظر هم‌ماه', period=TREND_PERIOD)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(monthly_reviews.index.to_timestamp(),monthly_reviews.mean_score,marker='o',color='#c17757')
ax.axhline(review_baseline,ls='--',color='#587d82',label=fa('مبنای همه نظرها'))
heading(ax,'10.3',f'امتیاز نظرها در بازه؛ خارج بازه کنار گذاشته شد، {len(core_reviews):,} نظر',TREND_PERIOD)
axes_labels(ax,'ماه ثبت نظر','میانگین امتیاز نظرها');ax.legend(fontsize=8);plt.xticks(rotation=35);complete()

The March score in table 10.6 is below the matched-period averages. Review texts and delivery status are useful for investigating the dip; the trend alone does not reveal the cause.


### 10.4. Merchandise Value and Category Scores

In chart 10.4 each point is a category meeting the sample minimum among delivered orders. For review candidacy, beyond the sample minimum, high value and a clear gap from the all-review average are also required; the threshold and the list are in tables 10.7 and 10.8. These rules are a first screening.


In [ ]:
category_value_threshold=category_eligible.value_brl.quantile(.75)
category_candidates=category_eligible.loc[(category_eligible.value_brl.ge(category_value_threshold)) &
                                           (category_eligible.score_gap.ge(SCORE_GAP))].sort_values('value_brl',ascending=False)
category_focus=category_eligible.loc[category_eligible.index.isin(['bed_bath_table','office_furniture','watches_gifts'])].copy()
category_focus['candidate']=category_focus.index.isin(category_candidates.index)
show_table('10.7',f'نامزد دسته؛ ارزش حداقل {category_value_threshold:,.2f} BRL، {MIN_REVIEWS} نظر و فاصله {SCORE_GAP:.2f}',
      category_candidates[['value_brl','review_records','mean_score','score_gap']],category_formats,category_labels, sample=f'{int(category_candidates.review_records.sum()):,} نظر–دسته منتخب', period=ORDER_PERIOD)
show_table('10.8',f'دسته‌های منتخب، حداقل {MIN_REVIEWS} نظر، و مقایسه با آستانه',
      category_focus[['value_brl','review_records','mean_score','score_gap','candidate']],
      category_formats,{**category_labels,'candidate':'در فهرست نامزدها'}, sample=f'{int(category_focus.review_records.sum()):,} نظر–دسته منتخب', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.8,4.8))
ax.scatter(category_eligible.value_brl,category_eligible.mean_score,s=35,alpha=.5,color='#5a9897',label=fa('دسته‌های واجد نمونه'))
ax.scatter(category_candidates.value_brl,category_candidates.mean_score,s=70,color='#c65f50',label=fa('نامزد بررسی'))
ax.axhline(review_baseline,color='#76888a',ls='--',label=fa('مبنای همه نظرها'))
ax.set_xscale('log')
heading(ax,'10.4',f'ارزش و امتیاز دسته‌ها، {len(category_eligible):,} دسته، حداقل {MIN_REVIEWS} نظر')
axes_labels(ax,'ارزش کالا، تحویل‌شده، BRL، محور لگاریتمی','میانگین امتیاز نظرها')
ax.legend(fontsize=8);complete()

In [ ]:
delivered_review_baseline = reviews.loc[reviews.order_id.isin(set(delivered_orders.order_id)), 'review_score'].mean()
baseline_sensitivity = pd.DataFrame({
    'gap_bed_bath_table': [review_baseline - category_metrics.loc['bed_bath_table', 'mean_score'],
                           delivered_review_baseline - category_metrics.loc['bed_bath_table', 'mean_score']],
    'gap_office_furniture': [review_baseline - category_metrics.loc['office_furniture', 'mean_score'],
                             delivered_review_baseline - category_metrics.loc['office_furniture', 'mean_score']],
}, index=['همه نظرها (مبنای جدول 10.1)', 'فقط نظرهای سفارش تحویل‌شده'])
baseline_sensitivity.index.name = 'baseline'
show_table('10.9', 'حساسیت فاصله امتیاز به انتخاب مبنا، دسته‌های منتخب', baseline_sensitivity,
      {c:'score' for c in baseline_sensitivity.columns},
      {'gap_bed_bath_table':'فاصله bed_bath_table','gap_office_furniture':'فاصله office_furniture'},
      sample=f'{len(reviews):,} نظر', period=ORDER_PERIOD)


Table 10.7 separates categories meeting both conditions from those near the average. Category score gaps are built on the "all reviews" basis (table 10.1); the alternative basis — only reviews of delivered orders — is higher and changes the gaps of categories near the 0.10 threshold. Table 10.9 measures this sensitivity for the two selected categories, and the borderline screening result for bed_bath_table should be read with it.

Focusing on eligible categories together with review volume reduces the chance of deciding on a very small difference.


### Categories with Small Samples

Table 10.10 lists categories that did not reach the review minimum and were excluded from the main ranking. Its purpose is to keep small-sample volume visible, not to provide a reliable ranking.


In [ ]:
category_small=category_metrics.loc[category_metrics.review_records.fillna(0).lt(MIN_REVIEWS)].sort_values('mean_score')
show_table('10.10',f'نمونه کم، کمتر از {MIN_REVIEWS} نظر',
      category_small[['review_records','reviewed_orders','mean_score','value_brl']],category_formats,category_labels, sample=f'{int(category_small.review_records.sum()):,} نظر–دسته کم‌نمونه', period=ORDER_PERIOD)

# 11. Delivery Delay & Customer Satisfaction

The customer-experience team hypothesises that delivery delay comes with lower customer satisfaction. Test this hypothesis with the data.

## Questions & Tasks

### 11.1. On-time vs Late Orders
1. Split delivered orders into "on-time" (early or on schedule) and "late" groups; compute the count and mean review score per group.
2. Show the two groups' mean scores in a comparison chart. How large is the gap? Are both groups large enough for a meaningful comparison?

### 11.2. Delay Severity
3. Split late orders into delay bands, for example 1–3 days, 4–7 days and more than 7 days; compute the mean score and order count per band.
4. Plot the mean score across the bands. Does the score fall regularly as the delay grows, or is there no clear pattern?

### 11.3. Differences Across Categories
5. Check whether the relationship is the same across product categories. Choose several categories with enough orders and compare on-time and late scores within them.

## Analytical Conclusion

Do the results support a relationship between delivery delay and customer satisfaction? What are the limits of this analysis? Explain why observing the relationship alone does not prove that delay is the cause of lower satisfaction.


### 11.1. Score and Delivery Status

Review records — without dropping duplicates — are joined to comparable delivered orders. Table 11.1 shows the three delivery groups "early", "on-time" and "late". Table 11.2 merges early and on-time for comparison with late, keeping review and reviewed-order counts separate.


In [ ]:
delivery_reviews=delivery[['order_id','delay_days','delivery_group','is_late']].merge(
    reviews[['order_id','review_score']],on='order_id',how='inner',validate='one_to_many')
group_reviews=(delivery_reviews.groupby('delivery_group').agg(review_records=('review_score','size'),
    reviewed_orders=('order_id','nunique'),mean_score=('review_score','mean')).reindex(group_keys))
group_reviews_display=group_reviews.copy();group_reviews_display.index=[group_labels[x] for x in group_reviews.index]
show_table('11.1','امتیاز نظرها و سه وضعیت تحویل',group_reviews_display,{'mean_score':'score'},
      {'review_records':'تعداد نظر','reviewed_orders':'سفارش نظردار','mean_score':'میانگین امتیاز نظرها'}, sample=f'{len(delivery_reviews):,} نظر', period=ORDER_PERIOD)
score_by_late=(delivery_reviews.groupby('is_late').agg(review_records=('review_score','size'),
    reviewed_orders=('order_id','nunique'),mean_score=('review_score','mean')).reindex([False,True]))
score_by_late['gap_score']=score_by_late.loc[False,'mean_score']-score_by_late.mean_score
score_by_late_display=score_by_late.copy();score_by_late_display.index=['زودتر و به‌موقع','دیرتر']
show_table('11.2','امتیاز نظرها، بدون تأخیر در برابر دیرتر',score_by_late_display,{'mean_score':'score','gap_score':'score'},
      {'review_records':'تعداد نظر','reviewed_orders':'سفارش نظردار','mean_score':'میانگین امتیاز نظرها',
       'gap_score':'فاصله از گروه بدون تأخیر'}, sample=f'{len(delivery_reviews):,} نظر', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.5,4))
ax.barh([fa(group_labels[x]) for x in group_keys],group_reviews.mean_score,color=['#4e9286','#8eb9ae','#ce7a5f'])
ax.invert_yaxis();ax.set_xlim(0,5.5)
for j,(_,row) in enumerate(group_reviews.iterrows()):
    ax.text(row.mean_score+.05,j,f'{row.mean_score:.2f}  ({int(row.review_records):,})',va='center',fontsize=9)
heading(ax,'11.1',f'امتیاز نظرها و تحویل، {len(delivery_reviews):,} نظر، کل دوره')
axes_labels(ax,'میانگین امتیاز نظرها','وضعیت تحویل');complete()

Each group's mean and sample size are in tables 11.1 and 11.2.

The score gap justifies reviewing late orders' experience; the relationship is not causal.


### 11.2. Delay Severity

Table 11.3 and chart 11.2 include only review records of late orders. Bands are calendar days after the estimated date. The score basis is the review level, as in table 11.2 — not a re-aggregation per order.


In [ ]:
late_reviews=delivery_reviews.loc[delivery_reviews.is_late].copy()
late_reviews['delay_band']=pd.cut(late_reviews.delay_days,bins=[0,3,7,np.inf],labels=['1-3','4-7','8+'])
severity=(late_reviews.groupby('delay_band',observed=True).agg(review_records=('review_score','size'),
    reviewed_orders=('order_id','nunique'),mean_score=('review_score','mean')).reindex(['1-3','4-7','8+']))
severity_display=severity.copy();severity_display.index=['\u200f1 تا 3 روز\u200f','\u200f4 تا 7 روز\u200f','\u200fبیش از 7 روز\u200f']
show_table('11.3','شدت تأخیر و امتیاز نظرها',severity_display,{'mean_score':'score'},
      {'review_records':'تعداد نظر','reviewed_orders':'سفارش نظردار','mean_score':'میانگین امتیاز نظرها'}, sample=f'{len(late_reviews):,} نظر', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.5,3.8))
ax.bar([fa(x) for x in severity_display.index],severity.mean_score,color='#c6795d')
heading(ax,'11.2',f'شدت تأخیر و امتیاز، {len(late_reviews):,} نظر، کل دوره')
axes_labels(ax,'بازه تأخیر، روز','میانگین امتیاز نظرها');ax.set_ylim(0,5);complete()

The mean score falls as the delay band grows; values with review and order counts are in table 11.3.

Longer delays have priority for process and customer-experience review, but other factors may also be involved.


### 11.3. Category Comparison

Tables 11.4 and 11.5 compare only categories that meet the minimum sample in each of the three delivery groups. Every score column carries its group's review count. The early–late gap is provided for comparison and is not necessarily a causal effect of delay.


In [ ]:
category_delivery=order_category.merge(delivery[['order_id','delivery_group']],on='order_id',how='inner',validate='many_to_one')
category_delivery=category_delivery.merge(reviews[['order_id','review_score']],on='order_id',how='inner',validate='many_to_many')
cat_group=category_delivery.groupby(['category','delivery_group']).agg(n=('review_score','size'),score=('review_score','mean'))
cat_n=cat_group['n'].unstack('delivery_group').reindex(columns=group_keys).fillna(0)
cat_score=cat_group['score'].unstack('delivery_group').reindex(columns=group_keys)
cat_three=cat_n.ge(MIN_REVIEWS).all(axis=1)
category_delivery_compare=pd.DataFrame({
    'early_n':cat_n.loc[cat_three,'early'],'early_score':cat_score.loc[cat_three,'early'],
    'on_time_n':cat_n.loc[cat_three,'on_time'],'on_time_score':cat_score.loc[cat_three,'on_time'],
    'late_n':cat_n.loc[cat_three,'late'],'late_score':cat_score.loc[cat_three,'late'],
    'gap_score':cat_score.loc[cat_three,'early']-cat_score.loc[cat_three,'late'],
}).sort_values('late_n',ascending=False)
show_table('11.4',f'امتیاز دسته‌ها، حداقل {MIN_REVIEWS} نظر در هر وضعیت',
      category_delivery_compare[['early_n','early_score','on_time_n','on_time_score']],
      {'early_score':'score','on_time_score':'score'},
      {'early_n':'نظر زودتر','early_score':'امتیاز زودتر','on_time_n':'نظر به‌موقع','on_time_score':'امتیاز به‌موقع'}, sample=f'{int(cat_n.loc[cat_three].to_numpy().sum()):,} نظر–دسته واجد شرط', period=ORDER_PERIOD)
show_table('11.5',f'امتیاز دسته‌های دیرتر، حداقل {MIN_REVIEWS} نظر در هر وضعیت',
      category_delivery_compare[['late_n','late_score','gap_score']],
      {'late_score':'score','gap_score':'score'},
      {'late_n':'نظر دیرتر','late_score':'امتیاز دیرتر','gap_score':'فاصله امتیاز'}, sample=f'{int(cat_n.loc[cat_three].to_numpy().sum()):,} نظر–دسته واجد شرط', period=ORDER_PERIOD)

# 12. Seller Performance

Management wants to know how sales are distributed across sellers and whether top sellers also keep customers satisfied.

## Questions & Tasks

### 12.1. Seller Count and Spread
1. Count unique sellers. Compute active sellers per state and compare the ten states with the most sellers.

### 12.2. Seller Sales
2. Compute items sold and sales value per seller; list the top ten by value and show them in a ranked chart.
3. Compute the top-ten sellers' share of total sales value. Is a large part of sales dependent on a few sellers?
4. Rank sellers by items sold. Is that ranking identical to the value ranking? Examine several important differences.

### 12.3. Sales and Customer Satisfaction
5. For sellers with enough reviews, compute the mean score. Plot sales against score so that high-sales low-score sellers stand out.
6. Pick sellers with high sales whose mean score is below the overall average; check their order or review counts to avoid conclusions from tiny samples.

## Business Conclusion

Is Olist's sales dependent on a few sellers? Which sellers, given their volume and review scores, may need further review?


### 12.1. Seller Count and State Distribution

Table 12.1 separates unique sellers in the data, sellers with items in all statuses and sellers with delivered items. Table 12.2 and chart 12.1 show seller locations; no current-activity assumption is made.


In [ ]:
seller_count=pd.DataFrame({'unique_sellers':[sellers.seller_id.nunique()],
    'sellers_with_items':[items.seller_id.nunique()],'sellers_delivered':[del_items.seller_id.nunique()]})
show_table('12.1','تعداد فروشندگان یکتا',seller_count,
      labels={'unique_sellers':'فروشنده یکتا','sellers_with_items':'دارای قلم در همه وضعیت‌ها',
              'sellers_delivered':'دارای قلم تحویل‌شده'}, sample=f'{len(sellers):,} فروشنده', period=ORDER_PERIOD)
seller_states=sellers.groupby('seller_state').seller_id.nunique().sort_values(ascending=False).head(10).to_frame('sellers')
show_table('12.2','ده ایالت نخست فروشندگان یکتا',seller_states,labels={'sellers':'تعداد فروشنده یکتا'}, sample=f'{len(sellers):,} فروشنده', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8,4.4))
ax.barh(seller_states.index,seller_states.sellers,color='#6a9fa4');ax.invert_yaxis()
heading(ax,'12.1',f'فروشندگان یکتا در ایالت‌ها، {len(sellers):,} فروشنده، کل دوره')
axes_labels(ax,'تعداد فروشنده یکتا','کلید ایالت');complete()

Table 12.2 shows the geographic concentration of unique sellers. That concentration alone does not determine shipping cost or efficiency.


### 12.2. Seller Value and Item Count

Tables 12.3, 12.4 and 12.5 place the value rank next to the item-count rank. Identifiers are shown in full in the tables and shortened in the chart.


In [ ]:
seller_top=seller_sales.sort_values('value_brl',ascending=False).head(10)
top10_seller_share=seller_top.value_brl.sum()/value_delivered*100
seller_top_summary=pd.DataFrame({'value_brl':[seller_top.value_brl.sum()],'share_pct':[top10_seller_share]})
show_table('12.3','ده فروشنده نخست بر اساس ارزش کالای تحویل‌شده',seller_top,
      {'value_brl':'brl'}, {'items':'تعداد کالا','orders':'تعداد سفارش','value_brl':'ارزش کالا، تحویل‌شده',
                             'item_rank':'رتبه تعداد','value_rank':'رتبه ارزش'}, sample=f'{len(del_items):,} قلم', period=ORDER_PERIOD)
show_table('12.4','سهم ده فروشنده نخست از ارزش کالای تحویل‌شده',seller_top_summary,
      {'value_brl':'brl','share_pct':'pct'},{'value_brl':'ارزش ده فروشنده','share_pct':'سهم ارزش کل'}, sample=f'{len(del_items):,} قلم', period=ORDER_PERIOD)
seller_compare_ids=seller_sales.nsmallest(10,'item_rank').index.union(seller_top.index)
seller_comparison=seller_sales.loc[seller_compare_ids].sort_values('item_rank').copy()
seller_comparison['rank_difference']=seller_comparison.value_rank-seller_comparison.item_rank
show_table('12.5','مقایسه رتبه تعداد کالا و ارزش فروشندگان',seller_comparison[['items','value_brl','item_rank','value_rank','rank_difference']],
      {'value_brl':'brl'}, {'items':'تعداد کالا','orders':'تعداد سفارش','value_brl':'ارزش کالا، تحویل‌شده',
                             'item_rank':'رتبه تعداد','value_rank':'رتبه ارزش','rank_difference':'اختلاف رتبه ارزش و تعداد'}, sample=f'{len(del_items):,} قلم', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.5,5))
ax.barh([str(x)[:8] for x in seller_top.index],seller_top.value_brl,color='#be895c');ax.invert_yaxis()
ax.xaxis.set_major_formatter(FuncFormatter(lambda v,pos:f'{v:,.0f}'))
heading(ax,'12.2',f'ده فروشنده نخست، شناسه کوتاه‌شده، {len(del_items):,} قلم، کل دوره')
axes_labels(ax,'ارزش کالا، سفارش‌های تحویل‌شده، BRL','شناسه کوتاه‌شده فروشنده');complete()

The seller with the most items does not necessarily have the highest value; table 12.5 shows each rank separately. Conclusion: sales is not dependent on a few sellers — the top ten hold 13.3% of delivered merchandise value (table 12.4) and the sales structure is distributed. Still, the candidate sellers of table 12.7 (such as 4a3ca931 with a 3.85 score and 7c67e144 with 3.50 against the 4.09 average) combine high volume with lower scores and deserve service-quality review first.


### 12.3. Sales and Score with a Minimum Sample

Every review belongs to the whole order; if an order has several sellers the same review appears under each related seller. Tables 12.6 and 12.7 and chart 12.3 apply the minimum sample. The value threshold in table 12.6 comes from the third quartile of eligible sellers — roughly a quarter of that group sits above it. The broad screen uses the third quartile of all sellers with delivered items and is separate from the strict list.


In [ ]:
order_seller=del_items[['order_id','seller_id']].drop_duplicates()
seller_review_rows=order_seller.merge(reviews[['order_id','review_score']],on='order_id',how='inner',validate='many_to_many')
seller_review=seller_review_rows.groupby('seller_id').agg(reviewed_orders=('order_id','nunique'),
    review_records=('review_score','size'),mean_score=('review_score','mean'))
seller_metrics=seller_sales.join(seller_review,how='left')
seller_metrics[['reviewed_orders','review_records']]=seller_metrics[['reviewed_orders','review_records']].astype('Int64')
seller_eligible=seller_metrics.loc[seller_metrics.reviewed_orders.ge(MIN_REVIEWS)].copy()
seller_threshold=seller_eligible.value_brl.quantile(.75)
seller_broad_threshold=seller_metrics.value_brl.quantile(.75)
seller_candidates=seller_eligible.loc[(seller_eligible.value_brl.ge(seller_threshold)) &
    (seller_eligible.mean_score.le(review_baseline-SCORE_GAP))].sort_values('value_brl',ascending=False)
seller_broad=seller_eligible.loc[(seller_eligible.value_brl.ge(seller_broad_threshold)) &
    (seller_eligible.mean_score.le(review_baseline-SCORE_GAP))].sort_values(['mean_score','value_brl'])
# پین‌کردن ردیف‌های مرجع گزارش: با تغییر نسخه داده این دو assert به‌عمد متوقف می‌شوند
# تا اعداد گزارش بدون بازبینی عوض نشوند.
reference_seller=seller_broad.index[0]
assert reference_seller == '1ca7077d890b907f89be8c954a02686a'
candidate_seller = str(seller_candidates.index[0])
assert candidate_seller == '4a3ca9315b744ce9f8e9374361493884'
assert {reference_seller, candidate_seller}.issubset(set(sellers.seller_id))
seller_screen=pd.DataFrame({'eligible':[len(seller_eligible)],'strict_brl':[seller_threshold],
    'strict_count':[len(seller_candidates)],'broad_brl':[seller_broad_threshold],'broad_count':[len(seller_broad)]})
show_table('12.6',f'آستانه‌ها؛ حداقل {MIN_REVIEWS} سفارش نظردار',seller_screen,
      {'strict_brl':'brl','broad_brl':'brl'},
      {'eligible':'فروشنده واجد شرط','strict_brl':'آستانه ارزش گروه واجد شرط',
       'strict_count':'تعداد نامزد سخت‌گیرانه','broad_brl':'آستانه ارزش همه فروشندگان تحویل‌شده',
       'broad_count':'تعداد نامزد گسترده'}, sample=f'{len(seller_eligible):,} واجد شرط و {len(seller_metrics):,} فروشنده', period=ORDER_PERIOD)
seller_selected=pd.concat([seller_candidates.head(3),seller_broad.loc[[reference_seller]]]).loc[
    :,['orders','value_brl','reviewed_orders','review_records','mean_score']]
show_table('12.7',f'فروشندگان منتخب، حداقل {MIN_REVIEWS} سفارش نظردار، و نمونه غربال گسترده',seller_selected,
      {'value_brl':'brl','mean_score':'score'},
      {'items':'تعداد کالا','orders':'تعداد سفارش','value_brl':'ارزش کالا، تحویل‌شده',
       'reviewed_orders':'سفارش نظردار','review_records':'تعداد نظر','mean_score':'میانگین امتیاز نظرها'}, sample=f'{len(seller_selected):,} فروشنده منتخب از {len(seller_eligible):,} واجد شرط', period=ORDER_PERIOD)
show_table('12.8','تعداد کالای فروشندگان منتخب',seller_metrics.loc[seller_selected.index,['items']],
      labels={'items':'تعداد کالای تحویل‌شده'}, sample=f'{len(seller_selected):,} فروشنده منتخب', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.5,4.8))
ax.scatter(seller_eligible.value_brl,seller_eligible.mean_score,s=23,alpha=.4,color='#5d9b9d',label=fa('فروشندگان واجد نمونه'))
ax.scatter(seller_candidates.value_brl,seller_candidates.mean_score,s=42,alpha=.8,color='#c86b53',label=fa('نامزد سخت‌گیرانه'))
ax.axhline(review_baseline,ls='--',color='#697f83',label=fa('مبنای همه نظرها'))
ax.set_xscale('log')
heading(ax,'12.3',f'ارزش و امتیاز فروشنده، {len(seller_eligible):,} فروشنده، حداقل {MIN_REVIEWS} سفارش نظردار')
axes_labels(ax,'ارزش کالا، تحویل‌شده، BRL، محور لگاریتمی','میانگین امتیاز نظرها')
ax.legend(fontsize=8);complete()

Candidate counts and thresholds are in table 12.6 and identifier details in table 12.7.

This list is for reviewing review texts and the order process; by itself it proves neither seller fault nor the cause of dissatisfaction.


# 13. Combined Analyses

Here several tables and variables are combined to answer questions that no single table can answer.

**At least three of the analyses below are required.**

## Analysis 1 — Best-selling categories with low satisfaction
Identify categories with high sales value whose mean score is below the overall average. Plot value against mean score, mark the target categories and explain why this group matters to the sales and customer-experience teams.

## Analysis 2 — Geography and delivery performance
Compute mean delivery time and the late-order percentage per state; identify states that underperform on both. Display results so that long-duration, high-delay regions separate from the rest, taking each region's order count into account.

## Analysis 3 — Product price and customer satisfaction
Compare sold-item prices with the review score of the related order; examine the relationship and draw a suitable chart. Do more expensive products get higher scores? Can we conclude that a higher price causes more satisfaction? Explain your answer.

## Analysis 4 — Payment amount and instalments
Examine payment amount and instalment count for orders; display the relationship and state whether higher amounts usually come with more instalments. Mind the difference between transaction count and unique orders.

## Analysis 5 — Category sales over time
Compute monthly sales value per category; pick several important categories and compare their sales over time. Which categories rose or fell significantly? Were the changes persistent or confined to a few months?

## Analysis 6 — Seller performance and customer satisfaction
Compute items sold, sales value and mean score per seller; identify high-volume low-score sellers. Show the results in a relationship chart and name at least three sellers for further review, taking review counts into account to avoid misleading comparisons.


### 13.1. High-Value, Low-Score Categories

Table 10.7 applies the value, sample-size and gap conditions simultaneously; table 10.8 compares the selected categories. This screen is a starting point for review and does not prove product defects.


### 13.2. Geography and Delivery

Chart 13.1 compares the mean duration and late rate of states meeting the sample minimum. Reference lines are the overall averages of comparable orders; each state's sample size is in table 13.1.


In [ ]:
overall_days=delivery.duration_days.mean()
overall_late=delivery.is_late.mean()*100
state_attention=state_eligible.loc[(state_eligible.mean_days.gt(overall_days))&
                                   (state_eligible.late_pct.gt(overall_late))].sort_values('orders',ascending=False)
show_table('13.1',f'ایالت‌های بالاتر از دو میانگین؛ حداقل {MIN_STATE} سفارش',state_attention,
      {'late_pct':'pct','mean_days':'decimal','median_days':'decimal'},
      {'orders':'سفارش قابل مقایسه','late':'سفارش دیرتر','mean_days':'میانگین مدت، روز',
       'median_days':'میانه مدت، روز','late_pct':'نرخ تأخیر'}, sample=f'{len(state_attention):,} ایالت منتخب از {len(state_eligible):,} واجد شرط', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8,5))
ax.scatter(state_eligible.mean_days,state_eligible.late_pct,s=45,alpha=.75,color='#4a9097')
ax.axvline(overall_days,ls='--',color='#7b8585',label=fa('میانگین کل مدت'))
ax.axhline(overall_late,ls=':',color='#c16e50',label=fa('نرخ تأخیر کل'))
heading(ax,'13.1',f'مدت و نرخ تأخیر ایالت‌ها، {len(state_eligible):,} ایالت، حداقل {MIN_STATE} سفارش')
axes_labels(ax,'میانگین مدت تحویل، روز','نرخ تأخیر، درصد');ax.legend(fontsize=8);complete()

States above both lines are listed in table 13.1. Rate and order volume together help prioritise routes; this comparison does not prove the cause of the differences.


### 13.3. Item Price and Order Score

Every delivered item is joined to the mean score of its order's reviews; in this section the score unit is the "order", separate from the "review" basis in table 10.1. Chart 13.2 and table 13.2 show the rank association.


In [ ]:
items_scored=del_items.merge(review_order,on='order_id',how='inner',validate='many_to_one').dropna(subset=['price','order_mean_score'])
price_rho=items_scored.price.rank().corr(items_scored.order_mean_score.rank())
price_relation=pd.DataFrame({'items':[len(items_scored)],'rank_correlation':[price_rho]})
show_table('13.2','رابطه قیمت کالا و میانگین امتیاز سطح سفارش',price_relation,
      {'rank_correlation':'decimal'},{'items':'تعداد قلم','rank_correlation':'همبستگی رتبه‌ای'}, sample=f'{len(items_scored):,} قلم', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.5,4.5))
price_plot=items_scored.loc[items_scored.price.gt(0)]
ax.scatter(price_plot.price,price_plot.order_mean_score,s=5,alpha=.025,color='#438b96')
ax.set_xscale('log')
heading(ax,'13.2',f'قیمت کالا و امتیاز سفارش، {len(price_plot):,} قلم، کل دوره')
axes_labels(ax,'قیمت کالا، BRL، محور لگاریتمی','میانگین امتیاز سطح سفارش');complete()

The coefficient and sample are in table 13.2.

Price–score correlation alone cannot prove that higher prices change satisfaction.


### 13.4. Payment Amount and Instalments at Order Level

Each order's transactions are summed and its maximum instalment count is used; multi-transaction orders are counted separately. Table 13.3 has a different unit from the transaction-level analysis in table 8.3.


In [ ]:
order_pay=payments.groupby('order_id').agg(total_brl=('payment_value','sum'),
    max_installments=('payment_installments','max'),transactions=('payment_value','size'))
order_pay_rho=order_pay.total_brl.rank().corr(order_pay.max_installments.rank())
multi_payment_orders=int(order_pay.transactions.gt(1).sum())
order_pay_summary=pd.DataFrame({'orders':[len(order_pay)],'multi_orders':[multi_payment_orders],
    'rank_correlation':[order_pay_rho]})
show_table('13.3','مبلغ و اقساط در سطح سفارش',order_pay_summary,{'rank_correlation':'decimal'},
      {'orders':'سفارش دارای پرداخت','multi_orders':'سفارش چندتراکنشی','rank_correlation':'همبستگی رتبه‌ای'}, sample=f'{len(order_pay):,} سفارش', period=ORDER_PERIOD)
fig,ax=plt.subplots(figsize=(8.2,4.5))
order_pay_plot=order_pay.loc[order_pay.total_brl.gt(0)]
ax.scatter(order_pay_plot.max_installments,order_pay_plot.total_brl,s=5,alpha=.04,color='#548e9a')
ax.set_yscale('log')
heading(ax,'13.3',f'مبلغ و اقساط سفارش، {len(order_pay_plot):,} سفارش با مبلغ مثبت، کل دوره')
axes_labels(ax,'بیشترین تعداد اقساط سفارش','مبلغ پرداخت سفارش، BRL، محور لگاریتمی');complete()

The coefficient and the count of multi-transaction orders are in table 13.3.

This comparison helps design payment experiments but does not show a causal effect of offering instalments.


### 13.5. Category Value over Time

Chart 13.4 compares the top five categories by delivered merchandise value inside the stated window. Table 13.4 totals the matched months of the two years and table 13.5 shows the percentage change for the same keys.


In [ ]:
top5_categories=top_value_categories.head(5).index.tolist()
category_month=del_items.loc[del_items.category.isin(top5_categories)&
    del_items.order_purchase_timestamp.ge(START)&del_items.order_purchase_timestamp.lt(END)].copy()
category_monthly=category_month.groupby(['purchase_month','category']).price.sum().unstack('category').reindex(columns=top5_categories,fill_value=0)
category_period=[]
for category in top5_categories:
    for year in (FIRST_YEAR,LAST_YEAR):
        values=category_month.loc[category_month.category.eq(category)&
                category_month.order_purchase_timestamp.ge(f'{year}-01-01')&
                category_month.order_purchase_timestamp.lt(f'{year}-09-01')]
        category_period.append({'category':category,'period':f'{year}-01 تا {year}-08',
                                'items':len(values),'value_brl':values.price.sum()})
category_year=pd.DataFrame(category_period).set_index(['category','period'])
show_table('13.4','ارزش دسته‌ها در ماه‌های مشترک',category_year,{'value_brl':'brl'},
      {'items':'تعداد کالا','value_brl':'ارزش کالا، تحویل‌شده'}, sample=f'{int(category_year["items"].sum()):,} قلم هم‌ماه', period=TREND_PERIOD)
category_year_pivot=category_year.value_brl.unstack('period')
category_year_pivot['change_pct']=(category_year_pivot['2018-01 تا 2018-08'].div(
    category_year_pivot['2017-01 تا 2017-08'])-1)*100
show_table('13.5','تغییر ارزش دسته‌ها در دوره‌های هم‌ماه',category_year_pivot,
      {'2017-01 تا 2017-08':'brl','2018-01 تا 2018-08':'brl','change_pct':'pct'},
      {'change_pct':'تغییر ارزش'}, sample=f'{int(category_year["items"].sum()):,} قلم هم‌ماه', period=TREND_PERIOD)
fig,ax=plt.subplots(figsize=(10,4.5))
for name in top5_categories:
    ax.plot(category_monthly.index.to_timestamp(),category_monthly[name],label=name,linewidth=1.7)
heading(ax,'13.4',f'ارزش ماهانه دسته‌ها، {len(category_month):,} قلم',TREND_PERIOD)
axes_labels(ax,'ماه ثبت سفارش','ارزش کالا، سفارش‌های تحویل‌شده، BRL')
ax.yaxis.set_major_formatter(FuncFormatter(lambda v,pos:f'{v:,.0f}'));ax.legend(fontsize=7,ncol=2);plt.xticks(rotation=35);complete()

Period differences and monthly volatility for each category are read from the table and chart.

A short-term change does not necessarily prove a persistent trend or a profit shift.


### 13.6. Sellers and Satisfaction

Table 12.7 introduces candidate sellers with full identifiers and reviewed-order counts, and table 12.8 shows their item counts. A broad-screen sample may sit below the strict value threshold. Review text belongs to the whole order and does not replace a formal investigation file.


# 14. Final Report for Management

Assume the analysis is finished and the results must be presented to Olist managers.

Managers are not interested in Python code; they want to know what the data says about the business and which actions can be proposed.

## 14.1. Business-State Summary

In a few paragraphs, explain the key results about: order trend, products, customers, delivery performance and customer satisfaction.

## 14.2. Five Key Findings

Choose five important findings. For each: state the result in plain language, give the number or indicator that supports it, add the chart that best shows the evidence, and explain why it matters to the business.

## 14.3. Three Recommendations for Management

Give three practical recommendations. For each explain: the issue being addressed, the finding that supports it, the change you expect the action to create, and which indicator should be monitored to evaluate the result.

**Note:** recommendations must be grounded in the analysis; avoid advice the available data does not support.


### Executive Summary

The answer to the four management questions is in the final synthesis. The five findings below place those indicators side by side with comparisons and managerial use.

**Finding 1 — Order and value trend:** matched-period orders went from 22,968 to 53,991 (+135.1%) and delivered merchandise value from 2,993,456.13 to 7,218,125.12 BRL; the busiest month is November 2017 with 7,544 orders. Tables 5.3 and 7.7 compare the matched periods and chart 5.2 shows the continuous window. Volume growth matters for capacity planning; its persistence is not guaranteed.

**Finding 2 — Customer concentration:** of 96,096 unique customers the top five states hold 77.1% and the city of São Paulo alone 15.6%. Table 6.2 and chart 6.1 show the spread. Concentration in the leading states makes regional service planning important.

**Finding 3 — Late rate:** the late rate went from 3.5% to 7.7% (+4.2 pp), reaching 19.0% in March 2018 and 12.4% in November 2017 alongside the sales peak. Table 9.6 and chart 9.3 show the rise. Each period's denominator is recorded and monitoring delivery routes is a priority.

**Finding 4 — Score and delivery:** the mean score of early/on-time orders' reviews is 4.29 (89,944 reviews) and of late orders 2.27 (6,409 reviews), falling to 2.10 for a 4–7 day delay. Table 11.2 compares the groups, table 11.3 splits severity and chart 11.1 shows the groups. This relationship does not prove the cause of the score gap.

**Finding 5 — Category and seller:** category office_furniture with 268,154.31 BRL value and a 3.64 score against the 4.09 average, and seller 4a3ca931 with 196,882.12 BRL value and a 3.85 score at a 30-review minimum, are review candidates. Table 10.8 shows the selected category's gap, table 12.7 introduces candidate sellers with the sample minimum and chart 7.2 shows category value ranks. These are starting points for order-experience review.


In [ ]:

print(f'سفارش در دو دوره هم‌ماه | {orders_2017:,} → {orders_2018:,} | {order_growth:.1f}%')
print(f'مشتری یکتا در SP / RJ | {state_summary.loc["SP","share_pct"]:.1f}% / {state_summary.loc["RJ","share_pct"]:.1f}%')
print(f'نرخ تأخیر در دو دوره هم‌ماه | {delivery_compare.iloc[0].late_pct:.1f}% → {delivery_compare.iloc[1].late_pct:.1f}%')
print(f'امتیاز نظرهای زودتر و به‌موقع / دیرتر | {score_by_late.loc[False,"mean_score"]:.2f} / {score_by_late.loc[True,"mean_score"]:.2f}')
print(f'امتیاز office_furniture / همه نظرها | {category_metrics.loc["office_furniture","mean_score"]:.2f} / {review_baseline:.2f}')


### Three Management Actions

**Delivery routes:** the rising late rate and state differences in tables 9.4 and 9.6 define the issue. A pilot on a selected route with a comparison group is monitored by order count, late rate, delay days and actual delivery duration. Expected change: the pilot route's late rate moving toward the 6.8% overall average.

**Category and seller:** the lower scores of the eligible category and sellers in tables 10.8 and 12.7 are the evidence. Reviewing review texts and running an improvement test beside a comparison group is monitored by review count, mean score and the share of low scores. Expected change: a higher mean score for the selected category and sellers and a smaller share of 1–2 star reviews.

**Customer return:** the second-order rate in table 6.5 defines the issue (1,827 of 77,482 equal-opportunity customers, 2.4%). A limited retention experiment beside a comparison group is monitored by second-order rate, group size and cost. Expected change: a higher 90-day second-order rate in the treatment group than in the control group.

### Strengths and Limitations

The share of early deliveries and of high scores are strengths; values and denominators are in tables 9.2 and 10.1.

- Delivered merchandise value excludes freight, costs and profit.
- Refund information and the cause of every payment difference are not available.
- Some categories, sellers and states are too small for comparison.
- Coverage of the first and last data months is uncertain.
- Not every order has a review and each review belongs to the whole order.
- The first recorded order is not necessarily the customer's first real purchase.
- A statistical relationship alone does not prove cause or the certain outcome of an action.
- Category score gaps use the all-reviews basis; on a delivered-orders-only basis the borderline gaps differ (table 10.9).


# 15. Chart & Results-Presentation Requirements

A chart is not needed for every question. A chart is warranted when it helps understand a distribution, compare groups, follow a time trend or uncover a relationship between variables.

Wherever a question asks you to examine a trend, compare groups or identify a relationship, use a chart that clearly answers it.

## Chart Requirements

- A clear title.
- Labelled axes.
- Units stated where needed.
- A chart type suited to the analysis goal and the data.
- After the chart, a few sentences describing the observed pattern.
- The business meaning of the pattern.

**Final goal:** not just drawing charts — charts must serve business questions and insight extraction.


### Answer to the Four Management Questions

**Business state:** orders and delivered merchandise value rise across the matched periods; tables 5.3 and 7.7 show the comparison.

**Customer behaviour:** state concentration and the equal-window second-order rate are in tables 6.2 and 6.5.

**Strengths:** the shares of early deliveries and of high scores are in tables 9.2 and 10.1.

**Areas to review:** the rising late rate and the score gap of late orders' reviews are recorded in tables 9.6 and 11.2.


# 16. Final Delivery

Deliver the following files at the end.

## 16.1. Jupyter Notebook

One `.ipynb` file containing: analysis code, essential outputs, charts, analytical commentary, results and business interpretations.

## 16.2. Management Report (PDF)

A PDF summarising the project's key findings and recommendations without programming code.

## 16.3. Final Project Review

Before delivery: run the notebook from the start; make sure cells execute in order; no execution errors; required charts and results are displayed; commentary matches the outputs; the management report is complete and readable.

## 16.4. Packaging

All essential project files must be inside a single ZIP archive. Do not send files separately.


### Computation Control

Table 16.1 checks totals, ranges, keys, delivery groups and seller identifiers.


In [ ]:

expected_categories = {'office_furniture','bed_bath_table','health_beauty','computers_accessories'}
original_keys = {'pc_gamer','portateis_cozinha_e_preparadores_de_alimentos'}
checks = {
 'ردیف‌های سفارش حفظ شده‌اند': len(orders)==len(raw['orders']),
 'ردیف‌های کالا حفظ شده‌اند': len(all_items)==len(raw['items']),
 'ردیف‌های نظر حفظ شده‌اند': len(reviews)==len(raw['reviews']),
 'تکرارهای جغرافیایی سازگارند': len(geo_unique)+int(geo.duplicated().sum())==len(geo),
 'ارزش فقط برای تحویل‌شده است': del_items.order_status.eq('delivered').all(),
 'جمع ارزش دسته‌ها سازگار است': np.isclose(category_sales.value_brl.sum(),value_delivered),
 'جمع ارزش فروشندگان سازگار است': np.isclose(seller_sales.value_brl.sum(),value_delivered),
 'مبنای امتیاز همه نظرها است': score_counts.reviews.sum()==len(reviews),
 'بازه سفارش با ثابت‌ها سازگار است': core_monthly.index.min()==START.to_period('M') and core_monthly.index.max()==(END-pd.Timedelta(days=1)).to_period('M'),
 'بازه تحویل با ثابت‌ها سازگار است': monthly_delivery.index.min()>=START.to_period('M') and monthly_delivery.index.max()<END.to_period('M'),
 'بازه نظر با ثابت‌ها سازگار است': monthly_reviews.index.min()>=START.to_period('M') and monthly_reviews.index.max()<END.to_period('M'),
 'تفکیک دوره‌های سفارش سازگار است': orders_2017+orders_2018+mid_orders==int(core_monthly.sum()),
 'گروه‌های تحویل سازگارند': ((delivery.delivery_group.eq('late'))==delivery.delay_days.gt(0)).all() and ((delivery.delivery_group.eq('on_time'))==delivery.delay_days.eq(0)).all() and ((delivery.delivery_group.eq('early'))==delivery.delay_days.lt(0)).all(),
 'حد نمونه دسته رعایت شده است': category_eligible.review_records.ge(MIN_REVIEWS).all(),
 'حد نمونه فروشنده رعایت شده است': seller_eligible.reviewed_orders.ge(MIN_REVIEWS).all(),
 'حد نمونه ایالت رعایت شده است': state_eligible.orders.ge(MIN_STATE).all(),
 'کلیدهای دسته در داده وجود دارند': expected_categories.issubset(set(category_sales.index)),
 'کلیدهای بدون ترجمه حفظ شده‌اند': original_keys.issubset(set(untranslated_categories.index)),
 'روش پرداخت نامشخص وجود دارد': 'not_defined' in set(payments.payment_type),
 'شناسه مرجع در داده وجود دارد': reference_seller=='1ca7077d890b907f89be8c954a02686a' and reference_seller in set(sellers.seller_id),
 'شناسه نامزد در داده وجود دارد': candidate_seller=='4a3ca9315b744ce9f8e9374361493884' and candidate_seller in set(sellers.seller_id),
 'شناسه‌های جدول فروشنده در داده‌اند': set(seller_selected.index).issubset(set(sellers.seller_id)) and set(seller_top.index).issubset(set(sellers.seller_id)),
 'پنجره سفارش دوم کامل است': (cohorts.first_order+pd.Timedelta(days=WINDOW_DAYS)<END).all(),
 'اجزای اختلاف پرداخت برابرند': np.isclose(payment_parts.difference_brl.sum(),payment_difference),
 'اختلاف پرداخت و تحویل‌شده درست است': np.isclose(payment_total-value_delivered,recon_total.delivered_gap_brl.iloc[0]),
 'تخصیص روش پرداخت سازگار است': np.isclose(recon_type.allocated_brl.sum(),payment_difference),
 'ترتیب ستون‌های امتیاز درست است': list(group_reviews.columns)==['review_records','reviewed_orders','mean_score'] and list(score_by_late.columns)==['review_records','reviewed_orders','mean_score','gap_score'],
 'اختلاف رتبه فروشنده درست است': seller_comparison.rank_difference.eq(seller_comparison.value_rank-seller_comparison.item_rank).all(),
 'امتیاز بازه تأخیر سازگار است': np.isclose(severity.loc['4-7','mean_score'],late_reviews.loc[late_reviews.delay_days.between(4,7),'review_score'].mean()),
}
assert all(checks.values()), [name for name, passed in checks.items() if not passed]
show_table('16.1','کنترل جمع‌ها، کلیدها و بازه‌ها',pd.Series(checks,name='passed').to_frame(),labels={'passed':'درست'}, sample=f'{len(checks):,} بررسی', period=ORDER_PERIOD)
report_payload={
 'date_min':str(orders.order_purchase_timestamp.min().date()),'date_max':str(orders.order_purchase_timestamp.max().date()),
 'orders_total':int(len(orders)),'orders_delivered':int(len(delivered_orders)),
 'orders_2017':orders_2017,'orders_2018':orders_2018,'order_growth':float(order_growth),
 'delivered_value':float(value_delivered),'all_status_value':float(value_all),
 'delivered_value_2017':float(delivered_period_values[0]),'delivered_value_2018':float(delivered_period_values[1]),
 'delivered_value_growth':float(delivered_growth),'delivered_items':int(len(del_items)),
 'customers':unique_customers,'state_sp_n':int(state_summary.loc['SP','customers']),
 'state_sp_pct':float(state_summary.loc['SP','share_pct']),
 'state_rj_n':int(state_summary.loc['RJ','customers']),'state_rj_pct':float(state_summary.loc['RJ','share_pct']),
 'top5_state_pct':float(top5_state_share),'single_customers':single_customers,
 'single_pct':float(single_customers/unique_customers*100),'cohort_customers':int(len(cohorts)),
 'cohort_repeat':int(cohorts.repeat_90d.sum()),'cohort_pct':float(repeat_90d_pct),
 'category_top5_pct':float(top5_category_share),'category_reference_threshold':float(category_value_threshold),
 'office_value':float(category_metrics.loc['office_furniture','value_brl']),
 'office_score':float(category_metrics.loc['office_furniture','mean_score']),
 'office_reviews':int(category_metrics.loc['office_furniture','review_records']),
 'office_reviewed_orders':int(category_metrics.loc['office_furniture','reviewed_orders']),
 'bed_value':float(category_metrics.loc['bed_bath_table','value_brl']),
 'bed_score':float(category_metrics.loc['bed_bath_table','mean_score']),
 'bed_reviews':int(category_metrics.loc['bed_bath_table','review_records']),
 'bed_gap':float(category_metrics.loc['bed_bath_table','score_gap']),
 'bed_candidate':bool('bed_bath_table' in category_candidates.index),
 'review_baseline':float(review_baseline),'reviews':int(len(reviews)),'score_max':int(reviews.review_score.max()),
 'good_pct':float(good_pct),'low_pct':float(low_pct),
 'review2017':float(review_compare.loc['2017-01 تا 2017-08','mean_score']),
 'review2018':float(review_compare.loc['2018-01 تا 2018-08','mean_score']),
 'review_march':float(review_compare.loc['2018-03','mean_score']),
 'review_march_n':int(review_compare.loc['2018-03','reviews']),
 'eligible_delivery':int(len(delivery)),'late_count':late_count,'late_pct':float(late_pct),
 'early_count':int(delivery_groups.loc['early','orders']),
 'early_pct':float(delivery_groups.loc['early','share_pct']),
 'on_time_count':int(delivery_groups.loc['on_time','orders']),
 'on_time_pct':float(delivery_groups.loc['on_time','share_pct']),
 'late_days_mean':float(late_mean),'late_over7':late_over7,
 'delay2017_n':int(delivery_compare.loc['2017-01 تا 2017-08','orders']),
 'delay2017_late':int(delivery_compare.loc['2017-01 تا 2017-08','late']),
 'delay2017_pct':float(delivery_compare.loc['2017-01 تا 2017-08','late_pct']),
 'delay2018_n':int(delivery_compare.loc['2018-01 تا 2018-08','orders']),
 'delay2018_late':int(delivery_compare.loc['2018-01 تا 2018-08','late']),
 'delay2018_pct':float(delivery_compare.loc['2018-01 تا 2018-08','late_pct']),
 'delay_change_pp':float(delay_change_pp),'march_delay_n':int(delivery_compare.loc['2018-03','orders']),
 'march_delay_pct':float(delivery_compare.loc['2018-03','late_pct']),
 'rj_delivery_n':int(state_eligible.loc['RJ','orders']),'rj_late_n':int(state_eligible.loc['RJ','late']),
 'rj_late_pct':float(state_eligible.loc['RJ','late_pct']),
 'al_delivery_n':int(state_eligible.loc['AL','orders']),'al_late_n':int(state_eligible.loc['AL','late']),
 'al_late_pct':float(state_eligible.loc['AL','late_pct']),
 'on_time_score':float(score_by_late.loc[False,'mean_score']),
 'on_time_reviews':int(score_by_late.loc[False,'review_records']),
 'on_time_orders':int(score_by_late.loc[False,'reviewed_orders']),
 'late_score':float(score_by_late.loc[True,'mean_score']),
 'late_reviews':int(score_by_late.loc[True,'review_records']),
 'late_review_orders':int(score_by_late.loc[True,'reviewed_orders']),
 'severity':{str(k):{'score':float(row.mean_score),'reviews':int(row.review_records),
                      'orders':int(row.reviewed_orders)} for k,row in severity.iterrows()},
 'payment_total':float(payment_total),'payment_difference':float(payment_difference),
 'payment_unavailable_diff':float(recon_status.loc['unavailable','difference_brl']),
 'payment_canceled_diff':float(recon_status.loc['canceled','difference_brl']),
 'payment_unavailable_no_item_brl':float(recon_status.loc['unavailable','paid_without_item_brl']),
 'payment_canceled_no_item_brl':float(recon_status.loc['canceled','paid_without_item_brl']),
 'unavailable_no_item':int(recon_status.loc['unavailable','no_item_orders']),
 'payment_card_pct':float(payment_summary.loc['credit_card','share_pct']),
 'payment_transactions':int(len(payments)),
 'reference_seller':reference_seller,'reference_seller_value':float(seller_metrics.loc[reference_seller,'value_brl']),
 'reference_seller_score':float(seller_metrics.loc[reference_seller,'mean_score']),
 'reference_seller_orders':int(seller_metrics.loc[reference_seller,'reviewed_orders']),
 'seller_candidate':candidate_seller,
 'seller_candidate_score':float(seller_candidates.iloc[0].mean_score),
 'seller_candidate_value':float(seller_candidates.iloc[0].value_brl),
 'seller_candidate_orders':int(seller_candidates.iloc[0].reviewed_orders),
 'seller_selected_count':int(len(seller_selected)),
 'trend_start':str(START.date()),'trend_end':str((END-pd.Timedelta(days=1)).date()),
 'delay_band_4_start':int(late_reviews.loc[late_reviews.delay_days.between(4,7),'delay_days'].min()),
 'delay_band_4_end':int(late_reviews.loc[late_reviews.delay_days.between(4,7),'delay_days'].max()),
 'continuous_orders':int(core_monthly.sum()),'shared_orders':int(orders_2017+orders_2018),
 'middle_orders':int(mid_orders),'continuous_months':int(len(core_monthly)),
 'core_delivery_count':int(len(core_delivery)), 'core_review_count':int(len(core_reviews)),
 'outside_review_count':int(len(reviews)-len(core_reviews)),
 'review_start':str(reviews.review_creation_date.min().date()),'review_end':str(reviews.review_creation_date.max().date()),
 'payment_delivered_gap':float(payment_total-value_delivered), 'payment_remainder':float(payment_residual),
 'delivered_missing_date':int(delivered_orders.order_delivered_customer_date.isna().sum()),
 'other_status_delivery_date':int((~orders.order_status.eq('delivered')&orders.order_delivered_customer_date.notna()).sum()),
 'not_defined_count':int(payments.payment_type.eq('not_defined').sum()),
 'min_reviews':MIN_REVIEWS,'min_state':MIN_STATE,'score_gap_threshold':SCORE_GAP,
}